# 파인튜닝 워크샵 (RunPod)

**LLM SFT · ETF Tool Calling · Embedding FT · Reranker FT 실습**

| Part | 주제 | 모델 | 데이터 | 핵심 평가 |
|---|---|---|---|---|
| **Part 1** | LLM SFT | `Qwen3-4B-Base` (QLoRA) | KoAlpaca | 한국어 ROUGE-L · eval loss |
| **Part 2** | ETF Tool Calling FT | `Qwen3-4B-Instruct-2507` | Glaive 사례 + ETF SQLite 합성 | Routing F1 · Argument · Execution |
| **Part 3** | Embedding FT | `BAAI/bge-m3` | ETF query/doc/qrels | MRR · nDCG · Recall · Matryoshka |
| **Part 4** | Reranker FT | `BAAI/bge-reranker-v2-m3` | Retriever hard negative | MAP · MRR · nDCG |

<br>

> **환경**: RunPod RTX 4090 24GB · Docker `unsloth/unsloth:latest` · Python 3.11 권장


### **학습 목표**

- QLoRA 기반 SFT가 Base 모델의 응답을 어떻게 바꾸는지 설명할 수 있습니다.
- Tool Calling에서 “호출 성공”과 “정답 실행”의 차이를 구분할 수 있습니다.
- Embedding과 Reranker의 역할, MRR·nDCG·Recall 지표를 읽을 수 있습니다.
- 학습이 끝났다는 사실과 모델이 개선됐다는 판단을 구분할 수 있습니다.

---

## 0. 환경 설정과 실행 프로파일

이 노트북은 기본값 `QUICK`으로 수업 시간 내 전체 흐름을 확인하고, `FULL`로 데이터와 평가 범위를 확장합니다.

- 패키지 설치 후 이미 import된 라이브러리 버전이 바뀌었다면 커널을 한 번 재시작하세요.
- Hugging Face 업로드는 기본 비활성입니다.
- ETF CSV는 노트북 기준 `data/etf_info.csv` 또는 `ETF_CSV_PATH` 환경변수로 지정합니다.


### **0-1. 필요한 패키지 설치**

- RunPod의 Unsloth 이미지에 없을 수 있는 평가·검색 패키지의 설치 명령을 모아 둡니다.
- `%pip`는 현재 Jupyter 커널이 사용하는 Python 환경에 패키지를 설치하는 IPython 명령입니다.


In [ ]:
# Unsloth Docker에 없는 학습·평가 패키지
# %pip install -q "sentence-transformers[train]>=5.2,<6" rouge-score kiwipiepy hf_transfer "sqlglot==30.12.0"

### **0-2. 실행 모드·재현성·GPU 조건 설정**

`(1) QUICK과 FULL`

| 항목 | QUICK | FULL | 의미 |
|---|---:|---:|---|
| SFT 최대 표본 | 3,000 | 10,000 | 학습에 사용할 최대 대화 수 |
| SFT 생성 평가 | 64 | 200 | Before/After에서 각각 생성할 일반 질문 수 |
| Embedding epoch | 2 | 3 | 전체 학습 데이터 반복 횟수 |
| 후보 문서 `K` | 20 | 50 | Reranker에 넘길 1차 검색 결과 수 |

QUICK은 전체 파이프라인이 연결되는지 빠르게 확인하는 교육용 실행입니다. 데이터와 평가 표본이 작으므로 **모델 배포나 승격을 결정하는 최종 실험이 아닙니다.**

`(2) 재현성 설정`

`SEED=42`를 Python과 NumPy에 함께 적용합니다. 같은 데이터 순서와 분할을 다시 만들기 위한 장치이지만, GPU 연산 전체가 언제나 비트 단위로 같아진다는 뜻은 아닙니다.

`(3) 하드웨어 검사`

CUDA GPU와 BF16 지원 여부를 먼저 검사해, 오래 학습한 뒤 환경 문제를 발견하는 일을 막습니다. BF16은 FP32보다 메모리를 적게 쓰면서 FP16보다 넓은 지수 범위를 제공하는 학습용 숫자 형식입니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. 필수 라이브러리 및 표준 모듈 로드
# -----------------------------------------------------------------------------
import ast  # Python 구문 트리(AST) 파싱용
import gc  # GPU/CPU 메모리 가비지 컬렉션(gc.collect())용
import hashlib  # 캐시 키 생성 및 데이터 해시 검증용
import importlib.metadata as metadata  # 패키지 설치 버전 확인용
import json  # JSON 데이터 포맷 처리용
import math  # 수학 계산 및 평가 지표 계산용
import numbers  # 수치형 데이터 타입 검사용
import os  # 환경 변수 접근 및 파일 경로 처리용
import random  # 난수 생성 (SEED 고정용)
import re  # 정규표현식 텍스트 파싱용
import sqlite3  # ETF 툴 호출 DB 실습용 SQLite 엔진
import warnings  # 실행 시 발생하는 경고 메시지 제어용
from collections import Counter, defaultdict  # 빈도수 집계 및 기본값 딕셔너리
from pathlib import Path  # 객체 지향적 파일 시스템 경로 다루기

import numpy as np  # 수치 연산 및 배열 처리
import pandas as pd  # ETF CSV 데이터 분석 및 표 형태 처리
import torch  # PyTorch 딥러닝 프레임워크 및 GPU 연산

# 불필요한 경고 메시지 출력 억제
warnings.filterwarnings("ignore")

# -----------------------------------------------------------------------------
# 2. 재현성(Reproducibility)을 위한 난수 시드 고정
# -----------------------------------------------------------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# -----------------------------------------------------------------------------
# 3. 실행 프로파일 설정 (QUICK: 교육용 시범 / FULL: 전체 학습 및 평가)
# -----------------------------------------------------------------------------
RUN_MODE = "QUICK"  # "QUICK" (빠른 테스트) | "FULL" (전체 실험)
MODE_CONFIG = {
    "QUICK": {
        "sft_max_samples": 3_000,      # SFT 학습 데이터 최대 샘플 수
        "sft_eval_n": 64,             # SFT 평가에 사용할 질문 수
        "sft_epochs": 1,              # SFT 학습 Epoch 수
        "tool_total": 3_000,          # Tool Calling 학습 합성 데이터 수
        "tool_eval_n": 120,           # Tool Calling 기본 평가 수
        "tool_challenge_n": 40,       # Tool Calling 난제/에지케이스 평가 수
        "etf_max_docs": 300,          # 검색용 ETF 문서 최대 수
        "ir_query_variants": 2,       # 질의 증강 변형 수
        "ir_challenge_n": 24,         # 검색 난제 평가 수
        "embedding_epochs": 2,        # Embedding 모델 학습 Epoch 수
        "ir_hard_negatives": 3,       # Reranker용 Hard Negative 추출 수
        "reranker_max_length": 256,   # Reranker 입력 최대 토큰 길이
        "candidate_k": 20,            # 1차 Retriever가 추출할 후보 문서 수
    },
    "FULL": {
        "sft_max_samples": 10_000,    # SFT 학습 데이터 최대 샘플 수
        "sft_eval_n": 200,            # SFT 평가에 사용할 질문 수
        "sft_epochs": 2,              # SFT 학습 Epoch 수
        "tool_total": 12_000,         # Tool Calling 학습 합성 데이터 수
        "tool_eval_n": 400,           # Tool Calling 기본 평가 수
        "tool_challenge_n": 80,       # Tool Calling 난제/에지케이스 평가 수
        "etf_max_docs": None,         # 검색용 ETF 문서 전체 사용
        "ir_query_variants": 4,       # 질의 증강 변형 수
        "ir_challenge_n": 64,         # 검색 난제 평가 수
        "embedding_epochs": 3,        # Embedding 모델 학습 Epoch 수
        "ir_hard_negatives": 7,       # Reranker용 Hard Negative 추출 수
        "reranker_max_length": 512,   # Reranker 입력 최대 토큰 길이
        "candidate_k": 50,            # 1차 Retriever가 추출할 후보 문서 수
    },
}
if RUN_MODE not in MODE_CONFIG:
    raise ValueError(f"RUN_MODE은 QUICK 또는 FULL이어야 합니다: {RUN_MODE}")
CFG = MODE_CONFIG[RUN_MODE]

# -----------------------------------------------------------------------------
# 4. 작업 디렉토리 및 Hugging Face Hub 업로드 설정
# -----------------------------------------------------------------------------
WORK_DIR = Path(os.getenv("WORK_DIR", "/workspace/finetuning"))
WORK_DIR.mkdir(parents=True, exist_ok=True)
PUSH_TO_HUB = False  # 학습 완료 후 HF Hub 자동 업로드 여부
HF_PRIVATE = True   # 업로드 시 비공개(Private) 레포지토리 지정 여부
HF_NAMESPACE = os.getenv("HF_NAMESPACE", "your-hf-name")  # HF 사용자/조직 이름

# -----------------------------------------------------------------------------
# 5. GPU 및 하드웨어 가속기(BF16 지원) 상태 검사
# -----------------------------------------------------------------------------
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU가 필요합니다. RunPod GPU Pod인지 확인하세요.")
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("이 강의 설정은 BF16 지원 GPU(Ampere 이후)가 필요합니다.")

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"MODE={RUN_MODE} | GPU={gpu_name} | VRAM={vram_gb:.1f} GB | BF16=True")

# 주요 파이썬 패키지 설치 버전 출력 (재현성 및 디버깅용)
for package in ["torch", "transformers", "datasets", "trl", "sentence-transformers", "accelerate", "sqlglot"]:
    try:
        print(f"  {package:<22} {metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"  {package:<22} not installed")


**실행 결과 설명**

- `MODE=QUICK`: 축소된 교육용 프로파일이 선택됐습니다.
- `NVIDIA GeForce RTX 4090`, `VRAM=23.5 GB`: 24GB급 단일 GPU가 인식됐습니다.
- `BF16=True`: 이 노트북의 혼합 정밀도 조건을 충족합니다.
- Torch `2.10.0+cu128`의 `cu128`은 CUDA 12.8 계열 빌드라는 뜻입니다.
- Transformers, TRL, Sentence Transformers, SQLGlot 버전은 결과 재현을 위한 provenance입니다.

모든 필수 패키지가 `not installed` 없이 출력됐고 하드웨어 검사 예외도 없으므로 환경 준비 단계는 통과했습니다.


### **0-3. Hugging Face 인증과 업로드 안전장치**

공개 모델을 내려받기만 할 때는 토큰이 없어도 됩니다. 비공개 모델 접근이나 Hub 업로드에는 토큰이 필요합니다.

- `HF_TOKEN`: 환경변수에서만 읽어 노트북에 비밀값을 직접 쓰지 않습니다.
- `PUSH_TO_HUB=False`: 실수로 외부 저장소에 모델을 업로드하지 않도록 기본 차단합니다.
- `HF_NAMESPACE`: 업로드할 계정을 명시합니다.

> 🔒 토큰 문자열은 출력하거나 Markdown에 붙여 넣지 않습니다.


In [ ]:
# Hugging Face 인증: 공개 모델 다운로드만 하면 토큰 없이도 진행 가능
from huggingface_hub import login

HF_TOKEN = os.getenv("HUGGINGFACE_TOKEN") or os.getenv("HF_TOKEN")
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("✅ Hugging Face 인증 완료")
else:
    print("ℹ️ HF 토큰 없음: 공개 다운로드만 사용합니다. Hub 업로드는 실행되지 않습니다.")

if PUSH_TO_HUB and (not HF_TOKEN or HF_NAMESPACE == "your-hf-name"):
    raise ValueError("Hub 업로드에는 HF_TOKEN과 HF_NAMESPACE 설정이 필요합니다.")


**실행 결과 설명**

`HF 토큰 없음`은 오류가 아닙니다. 사용한 모델과 데이터가 공개되어 다운로드가 가능했고 `PUSH_TO_HUB=False`라 업로드도 수행하지 않았습니다. 비공개 리소스가 필요할 때만 환경변수로 토큰을 주입합니다.


---

# Part 1: LLM SFT — `Qwen3-4B-Base`

**목표**: Base 모델에 instruction-following과 ChatML 대화 구조를 주입합니다.

데이터 보강 포인트:

1. 원본 순서의 앞부분만 자르지 않고 seeded shuffle 후 샘플링
2. 질문 정규화 기준 중복 제거와 품질 필터 통계
3. train/validation/test를 분리해 test는 최종 생성 평가에만 사용
4. Before/After 생성 조건을 greedy로 고정해 sampling 변동 제거
5. ROUGE-L과 함께 사실성 키워드 micro-set, 반복 4-gram, 중복 문장 비율로 회귀 진단

> ROUGE-L은 reference overlap이지 사실성 점수가 아닙니다. 두 축이 반대로 움직이면 정성적인 사례와 factual diagnostic을 우선 확인합니다.


> 📌 **참고**: 
> 여기서 말하는 회귀 진단은 **"모델을 파인튜닝한 후, 기존 성능이 이상하게 퇴보하거나 반복 출력 등 나쁜 덤핑 부작용이 생기지 않았는지 점검(Regression Test)하는 단계"** 를 뜻합니다.


### 1.1 ChatML과 response-only loss

```text
<|im_start|>system
당신은 도움이 되는 AI 어시스턴트입니다.<|im_end|>
<|im_start|>user
한국의 수도는?<|im_end|>
<|im_start|>assistant
대한민국의 수도는 서울입니다.<|im_end|>
```

`train_on_responses_only`는 user 구간을 `-100`으로 마스킹하고 assistant 응답 토큰에만 loss를 계산합니다. Base와 Instruct 모델은 역할이 다르므로 Part 1은 Base, Part 2는 이미 대화 능력이 있는 Instruct에서 시작합니다.


### **1-1. Base 모델에 QLoRA 어댑터 붙이기**

`(1) Base와 Instruct의 차이`

- **Base 모델**은 다음 토큰 예측을 배웠지만, 질문에 친절하게 답하는 대화 규칙은 충분히 학습하지 않았습니다.
- **Instruct 모델**은 사용자 지시와 대화 형식을 추가 학습한 모델입니다.
- Part 1은 `Qwen3-4B-Base`에 ChatML 형식과 응답 습관을 주입하는 SFT 실험입니다.

`(2) QLoRA 직관`

원본 가중치 $W$를 4-bit로 저장하고 고정한 채, 작은 저랭크 행렬만 학습합니다.

$$W' = W + \Delta W, \qquad \Delta W = BA$$

여기서 rank `r=16`은 $A$와 $B$ 사이의 좁은 통로 크기입니다. rank가 커지면 표현력과 학습 파라미터·메모리 사용량이 함께 늘어납니다.

`(3) 주요 설정`

- `max_seq_length=2048`: 한 학습 예제가 사용할 수 있는 최대 토큰 길이
- `load_in_4bit=True`: 원본 모델을 4-bit로 양자화해 VRAM 절감
- `target_modules`: attention과 MLP의 projection 층에 LoRA 적용
- gradient checkpointing: 중간 활성값을 다시 계산해 메모리 절감


In [ ]:
# -----------------------------------------------------------------------------
# Unsloth 라이브러리 및 데이터셋 관련 모듈 임포트
# -----------------------------------------------------------------------------
import unsloth
from datasets import Dataset, DatasetDict, load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# 모델 설정값: 최대 토큰 시퀀스 길이 및 학습 대상 Base 모델 경로
MAX_SEQ_LENGTH = 2048
SFT_MODEL = "unsloth/Qwen3-4B-Base"

# -----------------------------------------------------------------------------
# 1. Base 모델 및 토커나이저 로드 (NF4 4-bit 양자화 적용)
# -----------------------------------------------------------------------------
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=SFT_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,        # QLoRA용 4-bit NF4 양자화 로드로 VRAM 절약
    load_in_8bit=False,       # 8-bit 양자화는 미사용
    full_finetuning=False,    # 전체 파라미터 파인튜닝 비활성화 (LoRA 어댑터 사용)
    token=HF_TOKEN,           # Hugging Face 토큰 (필요 시 인증)
)

# Qwen3 Instruct 포맷에 맞는 대화 챗 템플릿(Chat Template) 적용
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

# -----------------------------------------------------------------------------
# 2. LoRA(Low-Rank Adaptation) 어댑터 부착 및 설정
# -----------------------------------------------------------------------------
model = FastLanguageModel.get_peft_model(
    model,
    r=16,  # LoRA Rank (어댑터 행렬 차원 크기)
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention Layer 레이어 모듈
        "gate_proj", "up_proj", "down_proj",     # MLP/FFN Layer 레이어 모듈
    ],
    lora_alpha=16,                        # LoRA 스케일링 계수 (r과 동일 시 스케일=1)
    lora_dropout=0,                       # Unsloth 최적화를 위해 0 권장
    bias="none",                           # 바이어스(bias) 파라미터 미학습
    use_gradient_checkpointing="unsloth",  # Unsloth 커스텀 그래디언트 체크포인팅으로 VRAM 절약
    random_state=SEED,                    # 가중치 초기화 난수 시드 고정
)

# -----------------------------------------------------------------------------
# 3. 학습 가능 파라미터 수 집계 및 출력
# -----------------------------------------------------------------------------
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ {SFT_MODEL} + LoRA r=16 | trainable={trainable/1e6:.1f}M/{total/1e6:.1f}M")

**실행 결과 설명**

- Unsloth가 Qwen3의 attention/MLP 36개 층을 패치했습니다.
- padding token이 없어 `<|PAD_TOKEN|>`을 지정했다는 메시지는 자동 보완 안내입니다.
- LoRA 학습 가능 파라미터는 약 **33.0M**으로 출력됐습니다.
- 수동 합계의 전체 2.465B와 Trainer가 뒤에서 표시하는 논리적 4.055B 차이는 4-bit packed storage 집계 방식 차이입니다.

`triton_kernels.routing` 메시지에는 `ERROR`가 쓰였지만 셀 자체는 성공했고 Qwen3 모델 로드·LoRA 부착·후속 학습이 모두 진행됐습니다. 따라서 이 실행에서는 GPT-OSS MoE용 optional kernel 진단으로 해석합니다.


> 📌 **참고**: : LoRA target_modules 7종

- **Self-Attention (4개: `q_proj`, `k_proj`, `v_proj`, `o_proj`)**: 토큰 간 관계 파악, 문맥 이해 및 챗 포맷 준수 개선
- **MLP / FFN (3개: `gate_proj`, `up_proj`, `down_proj`)**: 도메인 지식 흡수, 키-값 기억 및 추론 표현력 향상
- **All-Linear 적용 이점**: 7개 선형 모듈 전체에 LoRA를 부착하면 어텐션만 튜닝할 때보다 도메인 학습 효율과 모델 표현 용량이 극대화됩니다.


> 📌 **참고**: 타겟 모듈 선택 전략 (Why All 7 Modules?)

* **과거 (LoRA 초기)**: 주로 `q_proj`, `v_proj` 2개만 튜닝하여 파라미터 수와 VRAM을 극도로 아꼈습니다.
* **현재 표준 (All-Linear LoRA)**: `q, k, v, o` + `gate, up, down` **7개 선형 레이어 전체**에 LoRA를 적용합니다.
  - **이유**: 어텐션만 튜닝하는 것보다 MLP까지 함께 튜닝할 때 **도메인 지식 흡수력과 성능 표현력(Capacity)이 월등히 높아집니다.**
  - Unsloth와 같은 QLoRA 최적화 라이브러리를 사용하면 7개 모듈 전체에 적용해도 메모리 부담이 거의 없습니다.


### **1-2. KoAlpaca 데이터 감사와 90/5/5 분할**

학습 전에 데이터 품질을 확인하지 않으면 모델이 중복·반복·너무 짧은 답변까지 그대로 모방합니다.

`원본 → 무작위 섞기 → 길이/반복 필터 → 질문 중복 제거 → train/validation/test 분리`

- **train 90%**: 가중치 학습
- **validation 5%**: 학습 중 checkpoint 선택
- **test 5%**: 마지막 결과 보고

`prompt_hash` 집합이 서로 겹치지 않는지 `assert`로 검사합니다. 같은 질문이 train과 test에 함께 있으면 암기한 답을 일반화 성능으로 오해할 수 있기 때문입니다.

> 💡 `repeated_ngram_ratio`는 같은 4단어 묶음이 얼마나 반복되는지, `duplicate_sentence_ratio`는 같은 문장이 얼마나 되풀이되는지 보는 간단한 품질 신호입니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. KoAlpaca 데이터셋 로드 및 무작위 셔플
# -----------------------------------------------------------------------------
raw_sft = load_dataset("beomi/KoAlpaca-v1.1a", split="train")
order = list(range(len(raw_sft)))
random.Random(SEED).shuffle(order)  # 데이터 무작위화를 위해 SEED 고정 셔플

SYSTEM_PROMPT = "당신은 도움이 되는 AI 어시스턴트입니다."
reject = Counter()        # 필터링 단계별 제거 건수 집계 카운터
seen_prompts = set()      # 중복 프롬프트 감지용 세트
clean_sft = []         # 필터링 통과한 고품질 SFT 샘플 저장 리스트

# -----------------------------------------------------------------------------
# 2. 데이터 정규화 및 품질 검사 함수 정의
# -----------------------------------------------------------------------------
def normalize_space(value):
    """공백 정규화: 연속된 공백/줄바꿈을 단일 공백으로 치환하고 앞뒤 공백 제거"""
    return re.sub(r"\s+", " ", str(value or "")).strip()

def repeated_ngram_ratio(text, n=4):
    """반복 n-gram 비율 계산: 답변 내에 4개 단어 묶음이 유의미하게 반복되는지 측정"""
    tokens = normalize_space(text).casefold().split()
    if len(tokens) < n:
        return 0.0
    ngrams = [tuple(tokens[index:index + n]) for index in range(len(tokens) - n + 1)]
    return 1.0 - len(set(ngrams)) / len(ngrams)  # 1.0에 가까울수록 중복 반복이 심함

def duplicate_sentence_ratio(text):
    """중복 문장 비율 계산: 답변 내 동일 문장이 되풀이되는 비율 측정"""
    sentences = [normalize_space(item).casefold() for item in re.split(r"[.!?\n]+", str(text))]
    sentences = [item for item in sentences if len(item) >= 10]  # 최소 10자 이상 문장만 검사
    if len(sentences) < 2:
        return 0.0
    return 1.0 - len(set(sentences)) / len(sentences)

# -----------------------------------------------------------------------------
# 3. 데이터 필터링 루프 (길이 검사, 중복/반복 제거, 해시 기반 데이타 감사)
# -----------------------------------------------------------------------------
for idx in order:
    sample = raw_sft[idx]
    instruction = normalize_space(sample.get("instruction"))
    context = normalize_space(sample.get("input"))
    answer = normalize_space(sample.get("output"))
    
    # [필터 1] 질문(instruction) 길이가 너무 짧은 경우 제외 (최소 4자 이상)
    if len(instruction) < 4:
        reject["short_instruction"] += 1
        continue
    # [필터 2] 답변(answer) 길이가 적정 범위를 벗어난 경우 제외 (20자 ~ 1,500자)
    if not 20 <= len(answer) <= 1_500:
        reject["answer_length"] += 1
        continue
    # [필터 3] 답변 내 4-gram 반복(45% 초과) 또는 중복 문장(40% 초과)이 심한 저품질 데이터 제외
    if repeated_ngram_ratio(answer) > 0.45 or duplicate_sentence_ratio(answer) > 0.40:
        reject["repetitive_answer"] += 1
        continue
    
    # 프롬프트 생성 (instruction + optional context)
    user = instruction + (f"\n\n{context}" if context else "")
    prompt_key = normalize_space(user).casefold()
    
    # [필터 4] 동일한 질문(프롬프트) 중복 수집 방지
    if prompt_key in seen_prompts:
        reject["duplicate_prompt"] += 1
        continue
    seen_prompts.add(prompt_key)
    
    # 정제된 샘플 저장 (SHA-256 해시를 이용해 오염 방지 및 대화 포맷 구조화)
    clean_sft.append({
        "prompt": user,
        "reference": answer,
        "prompt_hash": hashlib.sha256(prompt_key.encode()).hexdigest(),  # 분할 시 교차 오염 검증용 해시
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
            {"role": "assistant", "content": answer},
        ],
    })
    if len(clean_sft) >= CFG["sft_max_samples"]:
        break

# -----------------------------------------------------------------------------
# 4. Dataset 생성 및 90/5/5 (Train / Validation / Test) 분할
# -----------------------------------------------------------------------------
sft_data = Dataset.from_list(clean_sft)
train_holdout = sft_data.train_test_split(test_size=0.10, seed=SEED)  # Train 90%, Holdout 10%
validation_test = train_holdout["test"].train_test_split(test_size=0.50, seed=SEED)  # Holdout을 5%:5%로 검증 및 테스트로 재분할
sft_split = DatasetDict({
    "train": train_holdout["train"],
    "validation": validation_test["train"],
    "test": validation_test["test"],
})

# -----------------------------------------------------------------------------
# 5. 프롬프트 해시 기반 데이터 오염(Data Leakage) 엄격 검증
# -----------------------------------------------------------------------------
hash_sets = {name: set(ds["prompt_hash"]) for name, ds in sft_split.items()}
# train 세트의 질문이 validation/test 세트에 섞이지 않았는지 교집합 검사 (isdisjoint)
assert hash_sets["train"].isdisjoint(hash_sets["validation"] | hash_sets["test"])
assert hash_sets["validation"].isdisjoint(hash_sets["test"])

# -----------------------------------------------------------------------------
# 6. 데이터 분할 및 정제 통계 출력
# -----------------------------------------------------------------------------
print(f"원본={len(raw_sft):,} | 채택={len(sft_data):,} | 제외={dict(reject)}")
for name, ds in sft_split.items():
    print(f"  {name:<10}: {len(ds):,}")


**실행 결과 설명**

원본 21,155건을 섞어 검사한 뒤 QUICK 한도 3,000건을 채택했습니다. 출력의 `answer_length: 2`는 채택 과정에서 답변 길이 조건으로 두 건을 제외했다는 뜻입니다.

- train 2,700 = 90%
- validation 150 = 5%
- test 150 = 5%

hash 불일치 assertion이 통과했고 세 split 크기가 출력됐으므로 질문 중복 기준의 분리 검사는 성공했습니다.


In [ ]:
# -----------------------------------------------------------------------------
# KoAlpaca 원본 샘플 vs 정제 및 챗 포맷 변환 후 샘플 비교 출력
# -----------------------------------------------------------------------------
raw_sample = raw_sft[0]          # KoAlpaca 원본 첫 번째 데이터
clean_sample = sft_split["train"][0]  # 정제 및 90/5/5 분할 후 train 첫 번째 데이터

print("=== [1. KoAlpaca 원본 데이터 구조] ===")
print(f"- instruction : {raw_sample.get('instruction')}")
print(f"- input       : {raw_sample.get('input')}")
print(f"- output      : {raw_sample.get('output')}")

print("\n=== [2. 정제 및 구조화 후 데이터 (Cleaned SFT Sample)] ===")
print(f"- prompt (User)      : {clean_sample['prompt']}")
print(f"- reference (Target) : {clean_sample['reference']}")
print(f"- prompt_hash        : {clean_sample['prompt_hash']}")

print("\n=== [3. 최종 LLM 학습용 대화 구조 (Qwen Chat Messages Format)] ===")
print(json.dumps(clean_sample["messages"], ensure_ascii=False, indent=2))


**비교 내용 요약**

1. **KoAlpaca 원본 데이터**: `instruction` (지시문), `input` (부가 맥락), `output` (답변) 형태의 키 구성
2. **최종 메시지 객체**: `system` / `user` / `assistant` 역할을 갖춘 Qwen 대화 템플릿용 JSON 메시지 구조

### **1-3. 학습 전 고정 생성 벤치마크 만들기**

`(1) 같은 테스트셋을 고정하는 이유`

Before와 After가 서로 다른 질문이나 다른 생성 옵션을 사용하면 학습 효과를 비교할 수 없습니다. 이 셀은 test 질문 64개와 사실성 진단 8개를 먼저 고정합니다.

`(2) greedy decoding`

`do_sample=False`는 매 단계 가장 점수가 높은 토큰을 선택합니다. 창의성은 줄지만 같은 조건의 Before/After 비교에는 유리합니다.

`(3) 한국어 ROUGE-L`

Kiwi로 한국어 형태소를 나눈 뒤, 정답과 생성문에 공통으로 등장하는 가장 긴 순서 부분열을 계산합니다. 문장 표현의 겹침을 보는 지표이지, 사실이 맞는지를 직접 증명하는 지표는 아닙니다.

In [ ]:
# -----------------------------------------------------------------------------
# 1. 한국어 형태소 분석기(Kiwi) 기반 ROUGE-L 스코어러 및 벤치마크 평가 세트 준비
# -----------------------------------------------------------------------------
from kiwipiepy import Kiwi
from rouge_score import rouge_scorer

class KoreanTokenizer:
    """Kiwi 형태소 분석기를 이용해 한국어 어휘 단위로 토큰화하는 커스텀 토커나이저"""
    def __init__(self):
        self.kiwi = Kiwi()
    def tokenize(self, text):
        return [token.form for token in self.kiwi.tokenize(text)]

# ROUGE-L: 가장 긴 공통 부분 수열(LCS) 기반 유사도 평가 (한국어 형태소 토큰 사용)
ko_scorer = rouge_scorer.RougeScorer(["rougeL"], tokenizer=KoreanTokenizer())

# 고정 SFT 평가셋 추출 (test split에서 설정된 sft_eval_n 개수만큼 추출)
sft_eval_examples = [
    {"q": row["prompt"], "ref": row["reference"]}
    for row in sft_split["test"].select(range(min(CFG["sft_eval_n"], len(sft_split["test"]))))
]

# -----------------------------------------------------------------------------
# 2. 사실성 및 부작용 점검용 고정 케이스 (Factual Regression Diagnostic)
#    - 파인튜닝 시 기존 상식/사실을 잊어버리거나 망각(Forgetting)하지 않는지 체크
# -----------------------------------------------------------------------------
FACTUAL_DIAGNOSTIC_CASES = [
    {"q": "대한민국의 수도는 어디인가요?", "required_any": ["서울"], "forbidden": []},
    {"q": "지구의 자연위성 이름은 무엇인가요?", "required_any": ["달"], "forbidden": []},
    {"q": "1기압에서 물이 끓는 섭씨 온도는 몇 도인가요?", "required_any": ["100"], "forbidden": []},
    {"q": "한글 창제를 주도한 조선의 왕은 누구인가요?", "required_any": ["세종"], "forbidden": []},
    {"q": "세계에서 면적이 가장 큰 대양은 무엇인가요?", "required_any": ["태평양"], "forbidden": []},
    {"q": "식물이 광합성에 사용하는 대표적인 기체는 무엇인가요?", "required_any": ["이산화탄소", "co2"], "forbidden": []},
    {"q": "무의 알싸한 매운맛을 내는 대표 성분 계열은 무엇인가요?", "required_any": ["이소티오시아네이트", "아이소티오시아네이트", "이소티오시안산"], "forbidden": ["캡사이신", "콜린"]},
    {"q": "2와 3을 더하면 얼마인가요?", "required_any": ["5"], "forbidden": []},
]

def factual_keyword_hit(text, case):
    """답변 텍스트 내 필수 키워드 포함 여부 및 금지 키워드 포함 여부 판별"""
    normalized = normalize_space(text).casefold()
    required = any(keyword.casefold() in normalized for keyword in case["required_any"])
    forbidden = any(keyword.casefold() in normalized for keyword in case["forbidden"])
    return bool(required and not forbidden)

# -----------------------------------------------------------------------------
# 3. SFT 평가용 문장 생성 함수 (Greedy Decoding / do_sample=False)
# -----------------------------------------------------------------------------
def generate_sft(question, max_new_tokens=256):
    """동일한 조건에서의 Before/After 비교를 위해 Greedy 디코딩 기반 추론 수행"""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")
    im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # 결정론적 생성 (Greedy decoding)
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=[tokenizer.eos_token_id, im_end_id],
        )
    generated = output[0][inputs["input_ids"].shape[-1]:]  # 입력 프롬프트를 제외한 생성 텍스트만 슬라이싱
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# -----------------------------------------------------------------------------
# 4. 파인튜닝 시작 전(Before SFT) 모델 생성 결과 미리 수집
# -----------------------------------------------------------------------------
FastLanguageModel.for_inference(model)  # Unsloth 추론 2배 고속화 모드 전환
sft_before = [generate_sft(item["q"]) for item in sft_eval_examples]
factual_before = [generate_sft(case["q"], max_new_tokens=128) for case in FACTUAL_DIAGNOSTIC_CASES]
print(f"✅ 고정 SFT test benchmark: {len(sft_eval_examples)}개 + factual diagnostic {len(factual_before)}개")


**실행 결과 설명**

일반 test 질문 64개와 factual diagnostic 8개의 Before 응답 생성이 완료됐습니다. 화면에 진행률이 없어도 다음 셀이 실행됐고 완료 메시지가 있으므로 정상입니다.

생성 호출은 학습 전 72회, 학습 후 다시 72회 수행됩니다. 최대 256토큰을 한 토큰씩 만드는 일반 test 64개가 가장 오래 걸릴 수 있으며, 첫 호출에는 CUDA/kernel warm-up 비용도 포함됩니다.


In [ ]:
# -----------------------------------------------------------------------------
# 파인튜닝 전(Before SFT) 모델 생성 결과 및 평가 데이터 샘플 확인
# -----------------------------------------------------------------------------
print("=== [1. 고정 평가 질문 및 파인튜닝 전 Base 모델 답변 샘플 (1번 예시)] ===")
print(f"- 질문 (Q)      : {sft_eval_examples[0]['q']}")
print(f"- Ground Truth  : {sft_eval_examples[0]['ref']}")
print(f"- Base 모델 답변: {sft_before[0]}")

print("\n=== [2. 사실성 진단(Factual Diagnostic) 케이스 및 Base 모델 생성 결과] ===")
for idx, (case, resp) in enumerate(zip(FACTUAL_DIAGNOSTIC_CASES[:3], factual_before[:3]), 1):
    hit = factual_keyword_hit(resp, case)
    print(f"[{idx}] Q: {case['q']}")
    print(f"    Required Keywords : {case['required_any']}")
    print(f"    Base Model Answer : {resp}")
    print(f"    Hit Result        : {'✅ PASS' if hit else '❌ FAIL'}")


### **1-4. ChatML 직렬화와 response-only loss**

대화 목록을 모델이 읽는 하나의 토큰열로 바꾸는 과정을 **직렬화**라고 합니다.

```text
system 지침 → user 질문 → assistant 정답
```

전체 토큰에 loss를 주면 모델이 사용자 질문까지 외우도록 학습할 수 있습니다. `train_on_responses_only`는 system/user 토큰의 label을 `-100`으로 바꾸고 assistant 응답 토큰만 loss 계산에 포함합니다.

- `n_tokens ≤ 2048`: 너무 긴 예제 제거
- device batch 8 × gradient accumulation 4 = **유효 배치 32**
- `eval_loss`: validation 응답 토큰의 예측 오차
- `load_best_model_at_end=True`: validation loss가 가장 낮은 checkpoint 복원


In [ ]:
# -----------------------------------------------------------------------------
# 1. ChatML 직렬화(Serialization) 및 토큰 길이 계산 함수
# -----------------------------------------------------------------------------
from trl import SFTConfig, SFTTrainer

def render_sft(batch):
    """대화 리스트(messages)를 Qwen 챗 템플릿 텍스트 스트링으로 렌더링 및 토큰 길이 집계"""
    texts = [
        tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        for messages in batch["messages"]
    ]
    lengths = [len(tokenizer(text, add_special_tokens=False)["input_ids"]) for text in texts]
    return {"text": texts, "n_tokens": lengths}

# 전체 Split(train/val/test) 데이터셋 텍스트 직렬화 및 토큰 세기
sft_text = sft_split.map(
    render_sft,
    batched=True,
    remove_columns=sft_split["train"].column_names,
)
before_sizes = {name: len(ds) for name, ds in sft_text.items()}

# -----------------------------------------------------------------------------
# 2. 최대 시퀀스 길이(MAX_SEQ_LENGTH=2048) 초과 데이터 필터링
# -----------------------------------------------------------------------------
sft_text = sft_text.filter(lambda row: row["n_tokens"] <= MAX_SEQ_LENGTH)
for name in sft_text:
    sft_text[name] = sft_text[name].remove_columns("n_tokens")
    print(f"  {name:<10}: {before_sizes[name]} → {len(sft_text[name])} (≤{MAX_SEQ_LENGTH} tokens)")

# -----------------------------------------------------------------------------
# 3. SFTTrainer 및 SFTConfig 파인튜닝 하이퍼파라미터 설정
# -----------------------------------------------------------------------------
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=sft_text["train"],
    eval_dataset=sft_text["validation"],
    args=SFTConfig(
        output_dir=str(WORK_DIR / "sft_checkpoints"),  # 체크포인트 저장 경로
        num_train_epochs=CFG["sft_epochs"],           # 학습 Epoch 수
        per_device_train_batch_size=8,                # 디바이스당 배치 크기 (8)
        gradient_accumulation_steps=4,                # 그래디언트 누적 단계 (4) -> 유효 배치 = 8x4 = 32
        learning_rate=2e-4,                           # 학습률 (LoRA 권장 2e-4)
        warmup_ratio=0.10,                            # 학습 초기 웜업 비율 (10%)
        optim="adamw_8bit",                           # 8-bit AdamW 옵티마이저로 VRAM 최적화
        lr_scheduler_type="cosine",                    # 코사인 학습률 스케줄러
        bf16=True,                                    # BF16 연산 가속 적용
        max_seq_length=MAX_SEQ_LENGTH,                # 최대 시퀀스 토큰 길이 (2048)
        logging_steps=10,                             # 10 스텝마다 로그 기록
        eval_strategy="epoch",                        # 매 Epoch 종료 시 평가
        save_strategy="epoch",                        # 매 Epoch 종료 시 체크포인트 저장
        save_total_limit=1,                           # 가장 최신/우수한 체크포인트 1개만 유지
        load_best_model_at_end=True,                  # 학습 종료 후 eval_loss가 가장 낮았던 모델 로드
        metric_for_best_model="eval_loss",            # 최고 모델 선발 기준 (검증 손실 낮음)
        greater_is_better=False,                      # eval_loss는 낮을수록 좋으므로 False
        report_to="none",                             # wandb 등 로깅 비활성화
        seed=SEED,                                    # 난수 시드 고정
        dataset_text_field="text",                    # 입력 텍스트 필드 지정
    ),
)

# -----------------------------------------------------------------------------
# 4. Response-only Loss 마스킹 적용 (Assistant 답변 토큰에만 Loss 계산)
# -----------------------------------------------------------------------------
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",       # User 질의 시작 태그 마커
    response_part="<|im_start|>assistant\n",     # Assistant 답변 시작 태그 마커
)

# -----------------------------------------------------------------------------
# 5. Assistant 학습 토큰 마스킹 정상 적용 여부 검증 (labels != -100)
# -----------------------------------------------------------------------------
masked = trainer.train_dataset[0]
learned = sum(label != -100 for label in masked["labels"])
assert learned > 0, "assistant 학습 토큰이 없습니다. chat template marker를 점검하세요."
print(f"✅ response-only label 확인: learned_tokens={learned}")


**실행 결과 설명**

세 split 모두 `2700→2700`, `150→150`으로 표시돼 2048토큰을 초과해 제거된 예제가 없습니다. `learned_tokens=158`은 첫 학습 예제에서 assistant 영역 158토큰이 실제 loss 대상이라는 확인입니다.


### **1-5. SFT 실행과 학습 로그 읽기**

**epoch**는 전체 train 데이터를 한 번 보는 단위이고, **step**은 optimizer가 가중치를 한 번 갱신하는 단위입니다. 2,700개 예제와 유효 배치 32이므로 한 epoch가 약 85 step이 됩니다.

- `train_loss`: 학습 데이터에서 계산한 평균 오차
- `eval_loss`: 보지 않고 남겨 둔 validation 데이터의 오차
- `peak_vram`: 실행 중 예약된 GPU 메모리의 최대값

loss는 낮을수록 대체로 좋지만, 서로 다른 모델·토큰화·loss 정의의 숫자를 직접 비교하면 안 됩니다. 또한 loss 감소가 사실성이나 검색 품질 향상을 자동으로 보장하지 않습니다.


In [ ]:
# -----------------------------------------------------------------------------
# Part 1: QLoRA 기반 Supervised Fine-Tuning (SFT) 학습 실행
# -----------------------------------------------------------------------------
print("🚀 Part 1 SFT 시작")

# Unsloth 전용 학습 모드로 전환 (그래디언트 계산 및 연산 최적화 커널 활성화)
FastLanguageModel.for_training(model)

# SFT 파인튜닝 학습 시작 (85 Steps / 1 Epoch 기준)
sft_stats = trainer.train()

# Validation 데이터셋 기반 검증 오차(eval_loss) 평가
sft_validation = trainer.evaluate()

# -----------------------------------------------------------------------------
# 최종 학습 결과(train_loss, eval_loss) 및 Peak VRAM 사용량 출력
# -----------------------------------------------------------------------------
print(f"train_loss={sft_stats.training_loss:.4f} | eval_loss={sft_validation['eval_loss']:.4f}")
print(f"peak_vram={torch.cuda.max_memory_reserved()/1024**3:.2f} GB")


**실행 결과 설명**

- 예제 2,700개, 1 epoch, 총 85 step
- device batch 8 × accumulation 4 × GPU 1 = 유효 배치 32
- 학습 가능 파라미터 33,030,144개, Trainer 기준 0.81%
- `train_loss=1.6925`, `eval_loss=1.7076`
- `peak_vram=19.02 GB`: 23.5GB GPU 범위 안에서 완료

eval loss가 train loss보다 약간 높은 것은 보지 않은 데이터가 더 어렵기 때문에 흔히 나타날 수 있습니다. 두 값만으로 과적합이나 응답 품질을 확정하지 않고 다음 생성 평가를 봅니다.


### **1-6. Before/After를 여러 지표로 함께 판정하기**

`(1) ROUGE-L F1`

최장 공통 부분열 길이를 $L$, 생성문 길이를 $m$, 정답 길이를 $n$이라 하면 다음처럼 정밀도와 재현율을 결합합니다.

$$P=\frac{L}{m},\quad R=\frac{L}{n},\quad F1=\frac{2PR}{P+R}$$

예를 들어 공통 토큰이 4개이고 생성문 5개, 정답 8개라면 $P=0.8$, $R=0.5$, $F1\approx0.615$입니다.

`(2) 회귀 진단`

- `factual_keyword_accuracy`: 8개 고정 질문에서 필수 키워드가 있고 금지 키워드가 없는 비율
- `empty_rate`: 빈 응답 비율
- `repeated_4gram_ratio`: 4-gram 반복 비율
- `duplicate_sentence_ratio`: 중복 문장 비율

작은 ROUGE 상승보다 사실 오류나 반복 증가가 더 심각할 수 있으므로, 표와 실제 생성 예시를 함께 읽어야 합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. SFT 파인튜닝 후(After SFT) 고정 평가 질문 및 사실성 질문 추론 생성
# -----------------------------------------------------------------------------
FastLanguageModel.for_inference(model)  # 추론 속도 최적화 모드로 전환
sft_after = [generate_sft(item["q"]) for item in sft_eval_examples]
factual_after = [generate_sft(case["q"], max_new_tokens=128) for case in FACTUAL_DIAGNOSTIC_CASES]

# -----------------------------------------------------------------------------
# 2. 한국어 ROUGE-L F1 점수 계산 및 파인튜닝 전/후 변화(Delta) 비교
# -----------------------------------------------------------------------------
rouge_rows = []
for item, before_text, after_text in zip(sft_eval_examples, sft_before, sft_after):
    before_score = ko_scorer.score(item["ref"], before_text)["rougeL"].fmeasure
    after_score = ko_scorer.score(item["ref"], after_text)["rougeL"].fmeasure
    rouge_rows.append({
        "question": item["q"],
        "before": before_score,
        "after": after_score,
        "delta": after_score - before_score,  # 점수 변화량 (양수=상승, 음수=하락)
    })

rouge_df = pd.DataFrame(rouge_rows)
rouge_summary = rouge_df[["before", "after", "delta"]].mean().to_dict()
print(pd.DataFrame([rouge_summary], index=["ROUGE-L macro"]).round(4))

# -----------------------------------------------------------------------------
# 3. 생성 품질(빈 응답율, 평균 길이, 4-gram 반복율, 중복 문장 비율) 진단 함수
# -----------------------------------------------------------------------------
def generation_quality_summary(texts):
    return {
        "empty_rate": float(np.mean([not normalize_space(text) for text in texts])),              # 빈 응답 비율
        "mean_chars": float(np.mean([len(text) for text in texts])),                             # 평균 글자 수
        "repeated_4gram_ratio": float(np.mean([repeated_ngram_ratio(text) for text in texts])), # 4개 단어 반복 비율
        "duplicate_sentence_ratio": float(np.mean([duplicate_sentence_ratio(text) for text in texts])), # 동일 문장 중복 비율
    }

# Before/After 간 생성 품질 및 사실성 정확도 비교 표 구성
quality_before = generation_quality_summary(sft_before)
quality_after = generation_quality_summary(sft_after)
factual_before_score = float(np.mean([factual_keyword_hit(text, case) for text, case in zip(factual_before, FACTUAL_DIAGNOSTIC_CASES)]))
factual_after_score = float(np.mean([factual_keyword_hit(text, case) for text, case in zip(factual_after, FACTUAL_DIAGNOSTIC_CASES)]))
diagnostic_table = pd.DataFrame([
    {"stage": "Before", "factual_keyword_accuracy": factual_before_score, **quality_before},
    {"stage": "After", "factual_keyword_accuracy": factual_after_score, **quality_after},
]).set_index("stage")
print("\nFactual/repetition regression diagnostic")
print(diagnostic_table.round(4).to_string())

# [성능 퇴보/부작용(Regression) 경고 제어 조건]
if factual_after_score < factual_before_score:
    print("⚠️ factual diagnostic이 하락했습니다. ROUGE 상승만으로 모델 개선을 결론내리지 마세요.")
if quality_after["repeated_4gram_ratio"] > quality_before["repeated_4gram_ratio"] + 0.02:
    print("⚠️ 반복 생성 비율이 증가했습니다. 데이터 품질·epoch·learning rate를 재검토하세요.")

# -----------------------------------------------------------------------------
# 4. 대표 답변 3건의 Before / After 정성적 샘플 비교 출력
# -----------------------------------------------------------------------------
for idx in range(min(3, len(sft_eval_examples))):
    print(f"\nQ: {sft_eval_examples[idx]['q'][:100]}")
    print(f"Before: {sft_before[idx][:160]}")
    print(f"After : {sft_after[idx][:160]}")

# -----------------------------------------------------------------------------
# 5. 메트릭 결과 JSON 파일 저장 및 SFT LoRA 어댑터/토커나이저 저장
# -----------------------------------------------------------------------------
sft_metrics_path = WORK_DIR / "sft_eval_metrics.json"
with sft_metrics_path.open("w", encoding="utf-8") as file:
    json.dump({
        "rouge_l": rouge_summary,
        "factual_keyword_accuracy": {"before": factual_before_score, "after": factual_after_score},
        "generation_quality": {"before": quality_before, "after": quality_after},
        "eval_n": len(sft_eval_examples),
    }, file, ensure_ascii=False, indent=2)

sft_save_path = WORK_DIR / "qwen3_sft_lora"
model.save_pretrained(str(sft_save_path))      # 파인튜닝된 LoRA 가중치 저장
tokenizer.save_pretrained(str(sft_save_path))  # 토커나이저 저장
if PUSH_TO_HUB:
    repo_id = f"{HF_NAMESPACE}/qwen3-4b-sft-lora"
    model.push_to_hub(repo_id, token=HF_TOKEN, private=HF_PRIVATE)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN, private=HF_PRIVATE)
print(f"✅ SFT LoRA={sft_save_path} | metrics={sft_metrics_path}")


**실행 결과 설명 — 왜 Part 1은 `FAIL`인가?**

| 지표 | Before | After | 판정 |
|---|---:|---:|---|
| ROUGE-L | 0.2322 | 0.2399 | +0.0077, 소폭 상승 |
| factual keyword accuracy | 0.875 | 0.875 | 변화 없음 |
| repeated 4-gram | 0.2391 | 0.3712 | 크게 악화 |
| duplicate sentence | 0.1336 | 0.1795 | 악화 |

After는 “하루에도 열두번씩”을 1960년대 시각 표기와 연결하고, 무의 매운맛을 캡사이신으로 설명하며 같은 문장을 반복했습니다. ROUGE의 작은 상승은 표현 겹침이 조금 늘었다는 뜻일 뿐 이 사실 오류를 상쇄하지 못합니다.

**결론**: 학습 실행과 adapter 저장은 성공했지만 품질 candidate는 실패입니다. 데이터 정제, learning rate, epoch, EOS·반복 억제 설정을 재검토해야 합니다.


### Part 1 실행 결과 해석 — `FAIL`

- 학습은 1 epoch, 85 step, 약 4분 45초에 끝났고 train loss 1.6925, eval loss 1.7076이었습니다.
- ROUGE-L은 0.2322→0.2399로 +0.0077 상승했지만, 64건 QUICK 표본의 작은 차이이므로 유의한 개선으로 단정할 수 없습니다.
- factual keyword accuracy는 0.875→0.875로 동일했습니다.
- repeated 4-gram ratio는 0.2391→0.3712, duplicate sentence ratio는 0.1336→0.1795로 악화됐습니다.
- “하루에도 열두번씩”의 유래를 임의로 1960년대·시각 표현과 연결하고, 무의 매운맛을 캡사이신으로 설명하는 등 환각·반복 사례가 확인됐습니다.

따라서 이 adapter는 ROUGE 소폭 상승보다 반복 회귀와 정성 실패가 더 중요하며, 개선 모델이 아닌 분석용 candidate로 판단해야 합니다.


---

# Part 2: ETF Tool Calling FT — Glaive 사례에서 실행 가능한 도메인 데이터로

Glaive는 다양한 함수와 대화가 들어 있는 범용 사례입니다. 이 노트북에서는 Glaive로 `<functioncall>` 변환과 데이터 정제를 설명하되, 실제 특화 데이터의 중심은 ETF CSV와 SQLite입니다.

도구 routing 규칙:

| 요청 | 도구 |
|---|---|
| 단일 ETF의 정확한 필드 | `get_etf_details` |
| 조건에 맞는 상세 행·정렬 | `search_etfs` |
| 명시된 ETF 2~5개 비교 | `compare_etfs` |
| 개수·평균·최솟값·그룹 집계 | `aggregate_etfs_sql` |
| 개념 설명·모호한 요청·실시간 가격/수익률 | 도구 없음 |

SQL은 assistant 본문이 아니라 `aggregate_etfs_sql`의 `sql` 인자 안에만 생성합니다.

평가는 합성기와 같은 분포의 **Template benchmark**와, 학습에 없던 표현·복합 조건·모호성·미지원 요청을 담은 **Challenge benchmark**를 분리합니다.


### **2-1. GPU 메모리 정리와 ETF 원천 데이터 준비**

Part 1 모델을 계속 GPU에 두면 다음 모델을 올릴 공간이 부족합니다. `del → gc.collect() → torch.cuda.empty_cache()` 순서로 Python 참조와 CUDA 캐시를 정리합니다.

이후 ETF CSV를 다음 흐름으로 정제합니다.

`경로 탐색 → 필수 열 검사 → 영문 열 이름 변환 → 결측/중복 제거 → 자산군 균형 표본 → entity split`

자산군별로 train/validation/test를 나누는 **층화 분할**은 특정 자산군이 한 split에만 몰리는 문제를 줄입니다. 같은 ETF 코드는 하나의 split에만 존재해야 합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Part 1 GPU 메모리(VRAM) 객체 해제 및 가비지 컬렉션
# -----------------------------------------------------------------------------
before_vram = torch.cuda.memory_allocated() / 1024**3
del model, trainer, tokenizer  # Part 1 모델 및 트레이너 객체 제거
gc.collect()                   # 파이썬 CPU 메모리 해제
torch.cuda.empty_cache()       # PyTorch GPU VRAM 캐시 정돈
print(f"🧹 VRAM {before_vram:.2f} → {torch.cuda.memory_allocated()/1024**3:.2f} GB")

# -----------------------------------------------------------------------------
# 2. ETF CSV 원천 데이터 파일 경로 자동 탐색 및 로드
# -----------------------------------------------------------------------------
CSV_CANDIDATES = [
    Path(os.getenv("ETF_CSV_PATH", "data/etf_info.csv")),
    Path.cwd() / "data" / "etf_info.csv",
    Path.cwd() / "llm-dev-prep" / "PRJ_04" / "Week1" / "data" / "etf_info.csv",
    WORK_DIR / "data" / "etf_info.csv",
    Path("/workspace/etf_info.csv"),
]
ETF_CSV_PATH = next((path for path in CSV_CANDIDATES if path.exists()), None)
if ETF_CSV_PATH is None:
    tried = "\n".join(f"  - {path}" for path in CSV_CANDIDATES)
    raise FileNotFoundError(
        "ETF CSV를 찾지 못했습니다. 저장소의 Week1/data/etf_info.csv를 RunPod에 업로드하거나 "
        f"ETF_CSV_PATH를 지정하세요.\n확인 경로:\n{tried}"
    )

# -----------------------------------------------------------------------------
# 3. 한글 ETF 컬럼명을 표준 영문 컬럼명으로 매핑 및 전처리
# -----------------------------------------------------------------------------
COLUMN_MAP = {
    "단축코드": "code",
    "한글종목명": "name_kr",
    "한글종목약명": "short_name_kr",
    "상장일": "listing_date",
    "기초지수명": "base_index",
    "추적배수": "tracking_multiple",
    "복제방법": "replication_method",
    "기초시장분류": "market",
    "기초자산분류": "asset_class",
    "운용사": "manager",
    "총보수": "total_fee",
    "과세유형": "tax_type",
}
raw_etf = pd.read_csv(ETF_CSV_PATH, encoding="cp949", dtype={"단축코드": str})
missing_columns = set(COLUMN_MAP) - set(raw_etf.columns)
if missing_columns:
    raise ValueError(f"ETF CSV 필수 열 누락: {sorted(missing_columns)}")

etf_df = raw_etf[list(COLUMN_MAP)].rename(columns=COLUMN_MAP).copy()
# 결측치 처리 및 수치 데이터 변환
text_columns = [column for column in etf_df.columns if column != "total_fee"]
for column in text_columns:
    etf_df[column] = etf_df[column].fillna("미상").astype(str).str.strip()
etf_df["code"] = etf_df["code"].str.upper()
etf_df["total_fee"] = pd.to_numeric(etf_df["total_fee"], errors="coerce")
etf_df = etf_df.dropna(subset=["code", "short_name_kr", "total_fee"])
etf_df = etf_df.drop_duplicates(subset=["code"], keep="last").reset_index(drop=True)

# -----------------------------------------------------------------------------
# 4. 자산군(asset_class) 균형 표본 추출 (QUICK 프로파일 300개 제한 시 사용)
# -----------------------------------------------------------------------------
def balanced_catalog_sample(frame, max_docs):
    """자산군별 분포 비율을 유지하며 지정된 max_docs 개수만큼 균형 샘플링"""
    if max_docs is None or len(frame) <= max_docs:
        return frame.copy()
    groups = list(frame.groupby("asset_class", sort=True))
    per_group = max(1, max_docs // len(groups))
    selected = []
    for _, group in groups:
        selected.append(group.sample(min(len(group), per_group), random_state=SEED))
    sampled = pd.concat(selected).drop_duplicates("code")
    remainder_n = max_docs - len(sampled)
    if remainder_n > 0:
        remainder = frame.loc[~frame.index.isin(sampled.index)]
        sampled = pd.concat([
            sampled,
            remainder.sample(min(remainder_n, len(remainder)), random_state=SEED),
        ])
    return sampled.sample(frac=1, random_state=SEED).reset_index(drop=True)

etf_df = balanced_catalog_sample(etf_df, CFG["etf_max_docs"])

# -----------------------------------------------------------------------------
# 5. 자산군 기준 Stratified Entity Split (train / validation / test 분할)
# -----------------------------------------------------------------------------
def add_entity_split(frame):
    """동일 ETF 종목(Entity)이 여러 Split에 겹치지 않도록 자산군별 70%/15%/15% 층화 분할"""
    result = frame.copy()
    result["split"] = "train"
    rng = np.random.default_rng(SEED)
    for _, group in result.groupby("asset_class"):
        indices = group.index.to_numpy().copy()
        rng.shuffle(indices)
        if len(indices) < 3:
            continue
        n_test = max(1, int(round(len(indices) * 0.15)))
        n_validation = max(1, int(round(len(indices) * 0.15)))
        if n_test + n_validation >= len(indices):
            n_test, n_validation = 1, 1
        result.loc[indices[:n_test], "split"] = "test"
        result.loc[indices[n_test:n_test + n_validation], "split"] = "validation"
    return result

etf_df = add_entity_split(etf_df)
assert etf_df["code"].is_unique  # 종목 코드 유일성 검사
assert set(etf_df["split"]) == {"train", "validation", "test"}

# -----------------------------------------------------------------------------
# 6. ETF 목록 최종 수량 및 자산군별 Split 교차 표(crosstab) 출력
# -----------------------------------------------------------------------------
print(f"✅ ETF catalog={len(etf_df):,} | path={ETF_CSV_PATH}")
print(pd.crosstab(etf_df["asset_class"], etf_df["split"]))


**실행 결과 설명**

VRAM 사용량이 `3.46→0.87 GB`로 줄어 다음 모델을 위한 공간을 확보했습니다. ETF catalog는 QUICK 한도인 300개이고, 자산군별 crosstab에서 train/validation/test가 모두 존재합니다.

주식처럼 원본 비중이 큰 자산군은 `86/19/19`, 부동산처럼 작은 자산군은 `9/2/2`입니다. 완전히 같은 크기는 아니지만 각 자산군이 평가 split에서 사라지는 문제를 줄였습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 정제 및 Stratified Split이 완료된 ETF 카탈로그 샘플 데이터 확인
# -----------------------------------------------------------------------------
print("=== [1. ETF 데이터프레임 무작위 3개 행 샘플] ===")
display(etf_df[["code", "short_name_kr", "asset_class", "manager", "total_fee", "split"]].sample(3, random_state=SEED))

print("\n=== [2. 첫 번째 ETF 종목의 상세 필드 구조] ===")
sample_etf_item = etf_df.iloc[0].to_dict()
print(json.dumps(sample_etf_item, ensure_ascii=False, indent=2))


### **2-2. split별 읽기 전용 SQLite 만들기**

Tool Calling은 모델이 만든 호출을 실제 데이터베이스에서 실행해 평가합니다. train용 tool 결과가 validation/test ETF 정보를 미리 보여주지 않도록 DB도 split별로 분리합니다.

- `train/validation/test.db`: 각 split의 학습·평가용 데이터
- `full.db`: 마지막 사용자 데모에서 전체 catalog 조회
- unique/index 생성: 코드 중복 방지와 검색 속도 개선
- 실행 시 `mode=ro`, `PRAGMA query_only=ON`: 읽기 전용 강제

> 💡 단순히 질문 텍스트만 나누는 것으로 충분하지 않습니다. tool 실행 결과가 다른 split의 정답을 노출하지 않는지도 확인해야 합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. SQLite 테이블 컬럼 목록 및 split별 DB 경로 정의
# -----------------------------------------------------------------------------
DB_COLUMNS = [
    "code", "name_kr", "short_name_kr", "listing_date", "base_index",
    "tracking_multiple", "replication_method", "market", "asset_class",
    "manager", "total_fee", "tax_type",
]
DB_PATHS = {
    split: WORK_DIR / f"etf_tools_{split}.db"
    for split in ["train", "validation", "test", "full"]
}

# -----------------------------------------------------------------------------
# 2. SQLite 데이터베이스 구축 및 색인(Index) 설정 함수
# -----------------------------------------------------------------------------
def build_etf_database(frame, path):
    """지정된 데이터프레임을 SQLite etfs 테이블로 저장하고 조회 성능 최적화용 인덱스 생성"""
    path.parent.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(path) as connection:
        # etfs 테이블 생성 및 데이터 삽입 (기존 테이블 존재 시 덮어쓰기)
        frame[DB_COLUMNS].to_sql("etfs", connection, if_exists="replace", index=False)
        # 빠른 조회 및 조인을 위한 인덱스 생성
        connection.execute("CREATE UNIQUE INDEX IF NOT EXISTS idx_etfs_code ON etfs(code)")        # ETF 코드 기본키 인덱스
        connection.execute("CREATE INDEX IF NOT EXISTS idx_etfs_manager ON etfs(manager)")         # 운용사 조회용 인덱스
        connection.execute("CREATE INDEX IF NOT EXISTS idx_etfs_asset ON etfs(asset_class)")     # 자산군 조회용 인덱스

# -----------------------------------------------------------------------------
# 3. Split별 독립된 SQLite DB 생성 (Data Leakage 방지)
# -----------------------------------------------------------------------------
# train/validation/test 각 Split에 해당하는 데이터만으로 독립된 DB 구축
for split in ["train", "validation", "test"]:
    build_etf_database(etf_df[etf_df["split"] == split], DB_PATHS[split])

# 사용자 데모 및 전체 조회를 위한 종합 full DB 구축
build_etf_database(etf_df, DB_PATHS["full"])
print("✅ SQLite:", {name: str(path) for name, path in DB_PATHS.items()})


**실행 결과 설명**

train·validation·test·full 네 SQLite 파일 경로가 출력됐습니다. 경로 생성과 인덱스 설정 중 예외가 없었으므로 DB 준비는 성공입니다. 이후 학습용 gold 결과는 해당 split DB에서만 만들고, 사용자 데모만 full DB를 사용합니다.


### **2-3. Tool Definition과 JSON Schema 이해하기**

모델은 Python 함수를 직접 실행하지 않습니다. 먼저 “어떤 도구를 어떤 JSON 인자로 호출할지” 구조화된 제안을 생성합니다.

| 도구 | 사용 상황 | 핵심 인자 |
|---|---|---|
| `get_etf_details` | 하나의 정확한 ETF 조회 | `identifier`, `fields` |
| `search_etfs` | 조건 검색·정렬 | 필터, 반환 열, 정렬, `limit` |
| `compare_etfs` | 2~5개 ETF 비교 | `identifiers`, `fields` |
| `aggregate_etfs_sql` | 개수·평균 등 집계 | 안전한 단일 `SELECT` |

`required`는 필수 인자, `enum`은 허용값 목록, `additionalProperties: false`는 정의하지 않은 인자를 거부한다는 뜻입니다. Schema는 모델 출력을 검증하는 계약이지 보안의 전부는 아닙니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. OpenAI / Qwen 호환 JSON Schema 열거형(Enum) 데이터 추출
# -----------------------------------------------------------------------------
FIELD_ENUM = [
    "code", "short_name_kr", "listing_date", "base_index", "tracking_multiple",
    "replication_method", "market", "asset_class", "manager", "total_fee", "tax_type",
]
ASSET_ENUM = sorted(etf_df["asset_class"].unique().tolist())      # 기초 자산 분류 열거형
MARKET_ENUM = sorted(etf_df["market"].unique().tolist())           # 기초 시장 분류 열거형
MULTIPLIER_ENUM = sorted(etf_df["tracking_multiple"].unique().tolist()) # 추적 배수 열거형

# -----------------------------------------------------------------------------
# 2. ETF 조회 및 집계를 위한 4개 Function/Tool 스키마(Schema) 정의
# -----------------------------------------------------------------------------
ETF_TOOLS = [
    # [Tool 1] get_etf_details: 단일 ETF 종목의 지정 필드 상세 조회
    {
        "type": "function",
        "function": {
            "name": "get_etf_details",
            "description": "종목코드 또는 정확한 ETF 약명으로 단일 ETF의 지정 필드를 조회합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "identifier": {"type": "string"},
                    "fields": {"type": "array", "items": {"type": "string", "enum": FIELD_ENUM}, "minItems": 1},
                },
                "required": ["identifier", "fields"],
                "additionalProperties": False,
            },
        },
    },
    # [Tool 2] search_etfs: 조건 필터링, 정렬 및 리스트 검색
    {
        "type": "function",
        "function": {
            "name": "search_etfs",
            "description": "ETF 상세 행을 조건 검색하거나 정렬합니다. 개수·평균 같은 집계에는 사용하지 않습니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keyword": {"type": "string"},
                    "manager": {"type": "string"},
                    "market": {"type": "string", "enum": MARKET_ENUM},
                    "asset_class": {"type": "string", "enum": ASSET_ENUM},
                    "tracking_multiple": {"type": "string", "enum": MULTIPLIER_ENUM},
                    "min_fee": {"type": "number"},
                    "max_fee": {"type": "number"},
                    "return_fields": {"type": "array", "items": {"type": "string", "enum": FIELD_ENUM}, "minItems": 1},
                    "sort_by": {"type": "string", "enum": ["short_name_kr", "listing_date", "total_fee"]},
                    "sort_order": {"type": "string", "enum": ["asc", "desc"]},
                    "limit": {"type": "integer", "minimum": 1, "maximum": 20},
                },
                "required": ["return_fields", "sort_by", "sort_order", "limit"],
                "additionalProperties": False,
            },
        },
    },
    # [Tool 3] compare_etfs: 명시된 2~5개 ETF 종목 간 비교
    {
        "type": "function",
        "function": {
            "name": "compare_etfs",
            "description": "사용자가 명시한 ETF 2~5개의 지정 필드를 나란히 비교합니다.",
            "parameters": {
                "type": "object",
                "properties": {
                    "identifiers": {"type": "array", "items": {"type": "string"}, "minItems": 2, "maxItems": 5},
                    "fields": {"type": "array", "items": {"type": "string", "enum": FIELD_ENUM}, "minItems": 1},
                },
                "required": ["identifiers", "fields"],
                "additionalProperties": False,
            },
        },
    },
    # [Tool 4] aggregate_etfs_sql: SQL 집계 쿼리(Count, Average 등)
    {
        "type": "function",
        "function": {
            "name": "aggregate_etfs_sql",
            "description": "ETF 개수·평균·합계·최솟값·최댓값 또는 그룹 집계가 필요할 때만 안전한 SQLite 집계 SELECT를 실행합니다.",
            "parameters": {
                "type": "object",
                "properties": {"sql": {"type": "string"}},
                "required": ["sql"],
                "additionalProperties": False,
            },
        },
    },
]

# -----------------------------------------------------------------------------
# 3. 정의된 Tool Definition 스키마 JSON 포맷 일부 출력
# -----------------------------------------------------------------------------
print(json.dumps(ETF_TOOLS, ensure_ascii=False, indent=2)[:2_000])


**실행 결과 설명**

출력은 전체 schema 중 앞 2,000자만 보여 줍니다. `get_etf_details`의 `identifier`와 `fields`, `search_etfs`의 필터·정렬·limit 계약이 JSON으로 직렬화된 것을 확인할 수 있습니다.


### **2-4. 실제 실행 전 다층 안전장치**

`(1) Structured tool`

필드명·정렬 방식·행 수를 allowlist로 제한하고 값은 `?` parameter binding으로 전달합니다. 값과 SQL 문법을 분리하므로 문자열 결합보다 SQL injection 위험을 줄입니다.

`(2) Aggregate SQL gateway`

SQLGlot으로 SQL을 AST로 파싱한 뒤 다음 조건을 검사합니다.

- 문장 하나와 `SELECT`만 허용
- `JOIN`, 서브쿼리, UNION, CTE 거부
- `etfs` 테이블과 허용 열만 사용
- 집계 함수가 반드시 존재
- 그룹 결과는 최대 20행, 전체 결과는 최대 100행

`(3) 결과의 의미 비교`

모델이 gold와 다른 SQL을 생성해도 결과 행이 같으면 **denotation equivalent**, 즉 실행 의미가 같다고 볼 수 있습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. SQLGlot 파서 임포트 및 허용 컬럼/정렬 컬럼 보안 화이트리스트 설정
# -----------------------------------------------------------------------------
import sqlglot
from sqlglot import exp

ALLOWED_COLUMNS = set(DB_COLUMNS)
SORTABLE_COLUMNS = {"short_name_kr", "listing_date", "total_fee"}

# -----------------------------------------------------------------------------
# 2. 안전한 SQLite 파라미터 바인딩 쿼리 실행 및 100행 제한 검사 함수
# -----------------------------------------------------------------------------
def normalize_scalar(value):
    """소수점 플로팅 연산 오차 방지를 위한 스칼라 수치 정규화 (8자리 반올림)"""
    if isinstance(value, float):
        return round(value, 8)
    return value

def fetch_rows(db_path, sql, params=()):
    """읽기 전용(RO) SQLite 커넥션을 통해 파라미터 바인딩 실행 및 100행 초과 차단"""
    uri = f"file:{Path(db_path).resolve()}?mode=ro"
    with sqlite3.connect(uri, uri=True) as connection:
        connection.execute("PRAGMA query_only=ON")  # DB 수정(INSERT/UPDATE/DELETE) 방지
        connection.set_progress_handler(lambda: 1, 200_000)  # 무한 루프 쿼리 타임아웃 차단
        cursor = connection.execute(sql, params)
        columns = [item[0] for item in cursor.description]
        rows = [[normalize_scalar(value) for value in row] for row in cursor.fetchmany(101)]
        if len(rows) > 100:
            raise ValueError("SQL 결과가 100행을 초과했습니다.")
    return {"columns": columns, "rows": rows}

def checked_fields(fields, include_code=True):
    """요청된 컬럼들의 보안 화이트리스트 검증 및 code 컬럼 자동 포함"""
    if not isinstance(fields, list) or not fields:
        raise ValueError("fields는 비어 있지 않은 배열이어야 합니다.")
    invalid = set(fields) - ALLOWED_COLUMNS
    if invalid:
        raise ValueError(f"허용되지 않은 fields: {sorted(invalid)}")
    result = list(dict.fromkeys(fields))
    if include_code and "code" not in result:
        result.insert(0, "code")
    return result

# -----------------------------------------------------------------------------
# 3. Structured Tools 구현: (1) 단일 조회 (2) 조건 검색 (3) 2~5개 비교
# -----------------------------------------------------------------------------
def get_etf_details(db_path, identifier, fields):
    """[Tool 1] 단일 ETF 종목코드 또는 약명으로 1개 레코드 정밀 조회"""
    fields = checked_fields(fields)
    cols = ", ".join(fields)
    sql = f"SELECT {cols} FROM etfs WHERE code = ? OR short_name_kr = ? LIMIT 1"
    return fetch_rows(db_path, sql, (str(identifier).upper(), str(identifier)))

def search_etfs(db_path, **arguments):
    """[Tool 2] 다중 필터(운용사, 시장, 자산군 등) 조건 검색 및 정렬/Limit 지정"""
    fields = checked_fields(arguments.pop("return_fields"))
    sort_by = arguments.pop("sort_by")
    sort_order = arguments.pop("sort_order").lower()
    limit = int(arguments.pop("limit"))
    if sort_by not in SORTABLE_COLUMNS or sort_order not in {"asc", "desc"}:
        raise ValueError("허용되지 않은 정렬 조건입니다.")
    if not 1 <= limit <= 20:
        raise ValueError("limit은 1~20이어야 합니다.")

    clauses, params = [], []
    for key in ["manager", "market", "asset_class", "tracking_multiple"]:
        if arguments.get(key) not in (None, ""):
            clauses.append(f"{key} = ?")
            params.append(arguments[key])
    if arguments.get("keyword"):
        clauses.append("(short_name_kr LIKE ? OR name_kr LIKE ? OR base_index LIKE ?)")
        kw = f"%{arguments['keyword']}%"
        params.extend([kw, kw, kw])
    if arguments.get("min_fee") is not None:
        clauses.append("total_fee >= ?")
        params.append(float(arguments["min_fee"]))
    if arguments.get("max_fee") is not None:
        clauses.append("total_fee <= ?")
        params.append(float(arguments["max_fee"]))
    if not clauses:
        raise ValueError("search_etfs에는 최소 한 개의 검색 조건이 필요합니다.")
    where = " AND ".join(clauses)
    cols = ", ".join(fields)
    sql = f"SELECT {cols} FROM etfs WHERE {where} ORDER BY {sort_by} {sort_order.upper()}, code ASC LIMIT ?"
    return fetch_rows(db_path, sql, (*params, limit))

def compare_etfs(db_path, identifiers, fields):
    """[Tool 3] 사용자가 지정한 2~5개 ETF 종목 비교 조회"""
    if not isinstance(identifiers, list) or not 2 <= len(identifiers) <= 5:
        raise ValueError("identifiers는 2~5개 배열이어야 합니다.")
    fields = checked_fields(fields)
    placeholders = ", ".join("?" for _ in identifiers)
    cols = ", ".join(fields)
    sql = f"SELECT {cols} FROM etfs WHERE code IN ({placeholders}) OR short_name_kr IN ({placeholders}) ORDER BY code"
    normalized = [str(item).upper() for item in identifiers]
    return fetch_rows(db_path, sql, (*normalized, *identifiers))

# -----------------------------------------------------------------------------
# 4. AST (Abstract Syntax Tree) 기반 SQL 게이트웨이 보안 검증기
# -----------------------------------------------------------------------------
def validate_aggregate_sql(sql):
    """SQLGlot AST 분석을 통해 JOIN, 서브쿼리, CTE 차단 및 집계함수(COUNT, AVG 등) 필수를 검증"""
    statements = sqlglot.parse(sql, read="sqlite")
    if len(statements) != 1 or not isinstance(statements[0], exp.Select):
        raise ValueError("단일 SELECT만 허용합니다.")
    tree = statements[0]
    if tree.args.get("with") or tree.args.get("with_"):
        raise ValueError("CTE는 허용하지 않습니다.")
    forbidden = (exp.Join, exp.Subquery, exp.Union, exp.Intersect, exp.Except)
    if any(tree.find(node_type) is not None for node_type in forbidden):
        raise ValueError("JOIN·서브쿼리·집합 연산은 허용하지 않습니다.")
    tables = {table.name.lower() for table in tree.find_all(exp.Table)}
    if tables != {"etfs"}:
        raise ValueError("etfs 테이블 하나만 사용할 수 있습니다.")
    projection_aliases = {projection.alias for projection in tree.expressions if projection.alias}
    columns = {column.name for column in tree.find_all(exp.Column)}
    invalid_columns = columns - ALLOWED_COLUMNS - projection_aliases
    if invalid_columns:
        raise ValueError(f"허용되지 않은 열: {sorted(invalid_columns)}")
    if not list(tree.find_all(exp.AggFunc)):
        raise ValueError("COUNT/AVG/MIN/MAX/SUM 집계가 필요합니다.")

    group = tree.args.get("group")
    group_columns = {column.name for column in group.find_all(exp.Column)} if group else set()
    for projection in tree.expressions:
        node = projection.this if isinstance(projection, exp.Alias) else projection
        if isinstance(node, exp.Column) and node.name not in group_columns:
            raise ValueError("집계 밖 SELECT 열은 GROUP BY에 있어야 합니다.")
        if not isinstance(node, (exp.Column, exp.AggFunc)):
            raise ValueError("SELECT에는 그룹 열 또는 직접 집계식만 허용합니다.")

    if group:
        limit = tree.args.get("limit")
        if limit is None:
            tree = tree.limit(20)
        else:
            expression = limit.expression
            if not isinstance(expression, exp.Literal) or int(expression.this) > 20:
                raise ValueError("그룹 집계 LIMIT은 20 이하여야 합니다.")
    return tree.sql(dialect="sqlite")

# -----------------------------------------------------------------------------
# 5. Tool Call 라우팅 실행기 및 결과 해시 함수
# -----------------------------------------------------------------------------
def execute_tool_call(name, arguments, db_path):
    """모델이 요청한 Tool Name에 따라 해당 Python 함수 및 파라미터 매핑 실행"""
    if name == "get_etf_details":
        return get_etf_details(db_path, **arguments)
    if name == "search_etfs":
        return search_etfs(db_path, **arguments)
    if name == "compare_etfs":
        return compare_etfs(db_path, **arguments)
    if name == "aggregate_etfs_sql":
        safe_sql = validate_aggregate_sql(arguments["sql"])
        return fetch_rows(db_path, safe_sql)
    raise ValueError(f"정의되지 않은 도구: {name}")

def result_hash(result):
    """Tool 실행 결과의 결과 의미론적 동일성(Denotation Equivalence) 비교용 SHA-256 해시 생성"""
    payload = json.dumps(result, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha256(payload.encode()).hexdigest()


**실행 결과 설명**

함수와 검증 규칙만 정의하므로 출력이 없는 것이 정상입니다. 후속 합성기와 평가기가 이 함수를 실제로 호출하며, 잘못된 SQL·열·정렬·행 수가 들어오면 `ValueError`로 차단됩니다.


### 2.1 Glaive 데이터 사례

Glaive의 장점은 다양한 함수 호출을 한 번에 볼 수 있다는 점입니다. 반면 문자열 안의 역할 마커, single-quote arguments, 깨진 JSON, tool-call이 없는 일반 대화가 섞여 있어 그대로 학습하면 안 됩니다.

아래 셀은 호출/비호출 사례를 분리하고 `<functioncall>`을 Qwen `<tool_call>`로 정규화합니다. 실제 혼합 비율은 전체의 10%만 사용합니다.


### **2-5. Glaive 원본을 안전한 대화 형식으로 정제하기**

Glaive에는 함수 호출, 일반 대화, 작은따옴표 Python dict, 깨진 JSON이 섞여 있습니다. 파서는 다음 순서로 처리합니다.

`역할 분리 → 균형 잡힌 {...} 추출 → JSON/리터럴 파싱 → <tool_call> 변환 → 해시 기반 split`

- `call`: assistant가 도구 호출을 만든 사례
- `no_call`: 도구를 쓰지 않고 답해야 하는 사례
- reject: 구조나 schema를 신뢰할 수 없어 제외한 사례

no-call 데이터가 없으면 모델은 모든 질문에서 무조건 도구를 호출하는 **over-call** 습관을 배울 수 있습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Glaive-v2 데이터셋 로드 및 태그 괄호/따옴표 밸런싱 추적 함수
# -----------------------------------------------------------------------------
raw_glaive = load_dataset("glaiveai/glaive-function-calling-v2", split="train")

def extract_balanced_object(text, marker):
    """<functioncall> 태그 내의 괄호({}) 깊이와 따옴표 이스케이프를 추적하여 올바른 JSON 블록 추출"""
    marker_index = text.find(marker)
    start = text.find("{", marker_index)
    if marker_index < 0 or start < 0:
        return None, None
    depth, quote, escaped = 0, None, False
    for index in range(start, len(text)):
        char = text[index]
        if escaped:
            escaped = False
            continue
        if char == "\\" and quote:
            escaped = True
            continue
        if quote:
            if char == quote:
                quote = None
            continue
        if char in {'"', "'"}:
            quote = char
        elif char == "{":
            depth += 1
        elif char == "}":
            depth -= 1
            if depth == 0:
                return text[start:index + 1], (start, index + 1)
    return None, None

# -----------------------------------------------------------------------------
# 2. Python Dict 리터럴 및 JSON 파싱 호환 파서
# -----------------------------------------------------------------------------
def parse_function_object(raw):
    """작은따옴표(')나 ast 리터럴 포맷의 불량 JSON 문자열을 표준 딕셔너리로 정규화"""
    try:
        value = json.loads(raw)
    except json.JSONDecodeError:
        value = ast.literal_eval(raw)
    arguments = value.get("arguments", {})
    if isinstance(arguments, str):
        try:
            arguments = json.loads(arguments)
        except json.JSONDecodeError:
            arguments = ast.literal_eval(arguments)
    return {"name": value.get("name", ""), "arguments": arguments}

# -----------------------------------------------------------------------------
# 3. Glaive 포맷 대화 파싱 및 Qwen <tool_call> 태그 표준화
# -----------------------------------------------------------------------------
def parse_glaive(sample):
    """Glaive 형태 대화를 Qwen 표준 <tool_call> JSON 태그 포맷으로 파싱 및 변환"""
    chat = str(sample.get("chat", ""))
    system = str(sample.get("system", "")).strip()
    if not chat.strip() or not system or not re.search(r"function|tool", system, re.I):
        return None
    messages = [{"role": "system", "content": system.removeprefix("SYSTEM:").strip()}]
    parts = re.split(r"(USER:|ASSISTANT:|FUNCTION RESPONSE:)", chat)
    role = None
    called = False
    for part in parts:
        part = part.strip()
        if part == "USER:":
            role = "user"
        elif part == "ASSISTANT:":
            role = "assistant"
        elif part == "FUNCTION RESPONSE:":
            role = "tool"
        elif role and part:
            content = part
            if role == "assistant" and "<functioncall>" in content:
                raw_object, span = extract_balanced_object(content, "<functioncall>")
                if not raw_object:
                    return None
                try:
                    function_call = parse_function_object(raw_object)
                except (ValueError, SyntaxError):
                    return None
                preamble = content[:content.find("<functioncall>")].strip()
                block = f"<tool_call>\n{json.dumps(function_call, ensure_ascii=False)}\n</tool_call>"
                content = f"{preamble}\n{block}" if preamble else block
                called = True
            messages.append({"role": role, "content": content})
    if len(messages) < 3 or messages[1]["role"] != "user":
        return None
    return {"messages": messages, "called": called}

# -----------------------------------------------------------------------------
# 4. 해시 기반 해시 스플릿 (Train 70% / Val 15% / Test 15%) 및 수집
# -----------------------------------------------------------------------------
scan_limit = 20_000 if RUN_MODE == "QUICK" else 80_000
glaive_pools = {split: {"call": [], "no_call": []} for split in ["train", "validation", "test"]}
conversion_rejects = Counter()
for sample in raw_glaive.select(range(min(scan_limit, len(raw_glaive)))):
    parsed = parse_glaive(sample)
    if parsed is None:
        conversion_rejects["parse_or_schema"] += 1
        continue
    first_user = next(message["content"] for message in parsed["messages"] if message["role"] == "user")
    schema_signature = parsed["messages"][0]["content"]
    group_key = normalize_space(schema_signature + "|" + first_user).casefold()
    bucket = int(hashlib.sha256(group_key.encode()).hexdigest()[:8], 16) % 100
    split = "train" if bucket < 70 else "validation" if bucket < 85 else "test"
    kind = "call" if parsed["called"] else "no_call"
    glaive_pools[split][kind].append(parsed)

# -----------------------------------------------------------------------------
# 5. Glaive 풀 구성 현황 및 파싱 결과 샘플 출력
# -----------------------------------------------------------------------------
print("Glaive pools:", {split: {kind: len(rows) for kind, rows in pools.items()} for split, pools in glaive_pools.items()})
print("Glaive rejects:", dict(conversion_rejects))
for kind in ["call", "no_call"]:
    example = glaive_pools["train"][kind][0]
    print(f"\n--- Glaive {kind} 사례 ---")
    print(json.dumps(example["messages"][:3], ensure_ascii=False, indent=2)[:1_200])


**실행 결과 설명**

20,000개를 스캔해 18,791개를 call/no-call pool로 변환했고 1,209개를 `parse_or_schema`로 제외했습니다. 이는 약 93.96% 변환 성공입니다.

출력 예시에서 call 사례는 `<functioncall>`이 Qwen 형식 `<tool_call>` JSON으로 바뀌었고, no-call 사례는 일반 assistant 답변을 유지합니다. reject를 억지로 복구하지 않은 것은 깨진 정답을 학습시키지 않기 위한 보수적 선택입니다.


### **2-6. 실행 검증된 ETF 합성 데이터 만들기**

정답 문장을 먼저 쓰는 대신, 실제 ETF 행에서 **gold plan**을 만들고 도구로 실행해 결과를 검증한 뒤 대화로 직렬화합니다.

```text
실제 ETF 행
  → 정답 도구와 인자
  → split 전용 DB에서 실행
  → 결과 hash 저장
  → user / assistant tool_call / tool / final answer 대화
```

학습 혼합에는 4개 도구 호출뿐 아니라 개념 질문, 지원하지 않는 실시간 요청, 모호한 요청, 일반 질문, Glaive call/no-call이 포함됩니다. `group_id`가 split 사이에 겹치지 않는지 검사해 같은 템플릿 그룹의 누수를 줄입니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Tool Call 및 일반 대화 합성 데이터셋 혼합 비율 설정 (TOOL_MIX)
# -----------------------------------------------------------------------------
TOOL_MIX = {
    "get_etf_details": 0.15,     # 단일 조회 (15%)
    "search_etfs": 0.15,          # 조건 검색 (15%)
    "compare_etfs": 0.10,         # 종목 비교 (10%)
    "aggregate_etfs_sql": 0.15,   # SQL 집계 (15%)
    "etf_concept": 0.10,          # 일반 ETF 개념 지식 (10% -> No Tool)
    "unsupported": 0.10,          # 미지원 실시간/수익률 질의 (10% -> No Tool 거절)
    "clarification": 0.05,       # 모호한 질의 재확인 (5% -> No Tool 요청)
    "general": 0.10,              # 일반 일상/계산 대화 (10% -> No Tool)
    "glaive_call": 0.05,          # Glaive 외부 도구 호출 사례 (5%)
    "glaive_no_call": 0.05,       # Glaive 외부 일반 대화 사례 (5%)
}
SPLIT_FRACTIONS = {"train": 0.70, "validation": 0.15, "test": 0.15}

TOOL_NAMES = {tool["function"]["name"] for tool in ETF_TOOLS}
PHRASE_OFFSET = {"train": 0, "validation": 1, "test": 2}

# -----------------------------------------------------------------------------
# 2. SQL 및 Qwen <tool_call> 태그 헬퍼 함수 정의
# -----------------------------------------------------------------------------
def sql_quote(value):
    """SQL 쿼리 내 따옴표 인젝션 방지 이스케이핑"""
    return "'" + str(value).replace("'", "''") + "'"

def assistant_tool_block(name, arguments):
    """Qwen3 표준 <tool_call> JSON 태그 블록 생성"""
    payload = {"name": name, "arguments": arguments}
    return f"<tool_call>\n{json.dumps(payload, ensure_ascii=False)}\n</tool_call>"

def result_preview(result):
    """Tool 실행 결과 딕셔너리를 텍스트 응답 프리뷰로 변환"""
    if not result["rows"]:
        return "조건에 맞는 ETF가 없습니다."
    preview = [dict(zip(result["columns"], row)) for row in result["rows"][:5]]
    return "조회 결과는 다음과 같습니다: " + json.dumps(preview, ensure_ascii=False)

# -----------------------------------------------------------------------------
# 3. Tool Call 사례 및 No Tool (일반 대화) 사례 객체 생성기
# -----------------------------------------------------------------------------
def make_call_case(split, action, user, arguments, group_id, case_index):
    """Gold Tool Call을 생성하고 해당 split 전용 DB에서 실제 실행/검증 후 대화 구성"""
    result = execute_tool_call(action, arguments, DB_PATHS[split])
    if result.get("rows") is None:
        raise AssertionError("gold tool result가 올바르지 않습니다.")
    messages = [
        {"role": "user", "content": user},
        {"role": "assistant", "content": assistant_tool_block(action, arguments)},
    ]
    # must-call 20% 사례는 tool response 후 grounded final answer까지 멀티턴 대화로 학습
    if case_index % 5 == 0:
        messages.extend([
            {"role": "tool", "content": json.dumps(result, ensure_ascii=False)},
            {"role": "assistant", "content": result_preview(result)},
        ])
    return {
        "source": "etf",
        "split": split,
        "action": action,
        "group_id": group_id,
        "user": user,
        "gold_arguments": arguments,
        "gold_result": result,
        "gold_result_hash": result_hash(result),
        "messages": messages,
        "tools": ETF_TOOLS,
    }

def make_no_call_case(split, category, user, answer, group_id):
    """도구 호출이 필요 없는(No-Tool) 일반 질문/개념/미지원 질의 사례 구성"""
    return {
        "source": "etf" if category != "general" else "general",
        "split": split,
        "action": "no_tool",
        "category": category,
        "group_id": group_id,
        "user": user,
        "gold_arguments": None,
        "gold_result": None,
        "gold_result_hash": None,
        "messages": [
            {"role": "user", "content": user},
            {"role": "assistant", "content": answer},
        ],
        "tools": ETF_TOOLS,
    }

def action_counts(total):
    """지정된 total 개수에 맞춰 액션 카테고리별 분량 집계"""
    counts = {name: int(total * fraction) for name, fraction in TOOL_MIX.items()}
    counts["etf_concept"] += total - sum(counts.values())
    return counts

def split_total(total, split):
    """Split별 수량(70% / 15% / 15%) 계산"""
    if split == "train":
        return int(total * 0.70)
    if split == "validation":
        return int(total * 0.15)
    return total - int(total * 0.70) - int(total * 0.15)

# -----------------------------------------------------------------------------
# 4. Action 및 Category별 Synthetic ETF 데이터 합성 엔진
# -----------------------------------------------------------------------------
def synthesize_action(split, action, count):
    """실제 ETF 카탈로그 데이터를 참조하여 템플릿 기반 질의 및 gold plan 생성"""
    frame = etf_df[etf_df["split"] == split].reset_index(drop=True)
    rows = []
    for index in range(count):
        row = frame.iloc[index % len(frame)]
        phrase = PHRASE_OFFSET[split]

        # [Action 1] 단일 상세 조회 (get_etf_details)
        if action == "get_etf_details":
            field_sets = [
                ["short_name_kr", "total_fee"],
                ["short_name_kr", "base_index", "manager"],
                ["short_name_kr", "tracking_multiple", "tax_type"],
            ]
            fields = field_sets[index % len(field_sets)]
            identifier = row["code"] if index % 2 == 0 else row["short_name_kr"]
            field_str = ", ".join(fields)
            templates = [
                f"{identifier}의 {field_str} 정보를 알려줘.",
                f"ETF {identifier} 상세 정보에서 {field_str}만 확인해줘.",
                f"{identifier} 상품의 {field_str} 값이 궁금해.",
            ]
            arguments = {"identifier": identifier, "fields": fields}
            rows.append(make_call_case(split, action, templates[phrase], arguments, f"get:{row['code']}:{field_str}", index))

        # [Action 2] 조건 검색 및 정렬 (search_etfs)
        elif action == "search_etfs":
            mode = index % 4
            base = {
                "return_fields": ["short_name_kr", "manager", "total_fee"],
                "sort_by": "total_fee",
                "sort_order": "asc",
                "limit": 5,
            }
            if mode == 0:
                arguments = {**base, "manager": row["manager"], "asset_class": row["asset_class"]}
                intent = f"{row['manager']}의 {row['asset_class']} ETF를 총보수 낮은 순으로 5개 보여줘."
            elif mode == 1:
                keyword = re.sub(r"^(KODEX|TIGER|ACE|SOL|KBSTAR|RISE|PLUS|HANARO)\s+", "", row["short_name_kr"], flags=re.I).split()[0]
                arguments = {**base, "keyword": keyword}
                intent = f"이름이나 지수에 {keyword}가 포함된 ETF를 찾아줘."
            elif mode == 2:
                arguments = {**base, "asset_class": row["asset_class"], "max_fee": float(row["total_fee"])}
                intent = f"{row['asset_class']} ETF 중 총보수 {row['total_fee']:.3f}% 이하 상품을 낮은 보수 순으로 보여줘."
            else:
                arguments = {**base, "market": row["market"], "tracking_multiple": row["tracking_multiple"]}
                intent = f"{row['market']} 시장의 추적배수 {row['tracking_multiple']} ETF를 찾아줘."
            templates = [intent, intent.replace("보여줘", "검색해줘"), intent.replace("찾아줘", "조회해줘")]
            rows.append(make_call_case(split, action, templates[phrase], arguments, f"search:{mode}:{row['code']}", index))

        # [Action 3] 종목 간 나란히 비교 (compare_etfs)
        elif action == "compare_etfs":
            same_asset = frame[frame["asset_class"] == row["asset_class"]]
            pool = same_asset if len(same_asset) >= 2 else frame
            sampled = pool.sample(min(3 if index % 2 else 2, len(pool)), random_state=SEED + index)
            identifiers = sampled["code"].tolist()
            fields = ["short_name_kr", "total_fee", "base_index"] if index % 2 else ["short_name_kr", "total_fee", "manager"]
            arguments = {"identifiers": identifiers, "fields": fields}
            joined = ", ".join(identifiers)
            field_str = ", ".join(fields)
            templates = [
                f"{joined} ETF의 {field_str}를 비교해줘.",
                f"ETF {joined} 사이에서 {field_str} 차이를 보여줘.",
                f"{joined} 상품을 {field_str} 기준으로 나란히 확인해줘.",
            ]
            group = "compare:" + ":".join(sorted(identifiers)) + ":" + field_str
            rows.append(make_call_case(split, action, templates[phrase], arguments, group, index))

        # [Action 4] SQL 통계 집계 (aggregate_etfs_sql)
        elif action == "aggregate_etfs_sql":
            mode = index % 6
            if mode == 0:
                sql = f"SELECT COUNT(*) AS etf_count FROM etfs WHERE asset_class = {sql_quote(row['asset_class'])}"
                user = f"{row['asset_class']} ETF는 모두 몇 개야?"
            elif mode == 1:
                sql = f"SELECT AVG(total_fee) AS average_fee FROM etfs WHERE manager = {sql_quote(row['manager'])}"
                user = f"{row['manager']} ETF의 평균 총보수는 얼마야?"
            elif mode == 2:
                sql = "SELECT asset_class, COUNT(*) AS etf_count FROM etfs GROUP BY asset_class ORDER BY etf_count DESC LIMIT 20"
                user = "기초자산별 ETF 개수를 비교해줘."
            elif mode == 3:
                sql = f"SELECT MIN(total_fee) AS minimum_fee FROM etfs WHERE market = {sql_quote(row['market'])}"
                user = f"{row['market']} 시장 ETF의 최저 총보수 값은?"
            elif mode == 4:
                threshold = float(row["total_fee"])
                sql = f"SELECT COUNT(*) AS etf_count FROM etfs WHERE total_fee <= {threshold:.6f}"
                user = f"총보수 {threshold:.3f}% 이하인 ETF는 몇 개야?"
            else:
                sql = "SELECT manager, AVG(total_fee) AS average_fee FROM etfs GROUP BY manager ORDER BY average_fee ASC LIMIT 20"
                user = "운용사별 평균 총보수를 낮은 순으로 비교해줘."
            variants = [user, user.replace("얼마야", "알려줘"), user.replace("비교해줘", "집계해줘")]
            arguments = {"sql": sql}
            rows.append(make_call_case(split, action, variants[phrase], arguments, f"sql:{split}:{mode}:{row['code']}:{result_hash({'sql': sql})[:12]}", index))

        # [Action 5] ETF 일반 개념 설명 (No Tool)
        elif action == "etf_concept":
            topics = [
                ("총보수", "총보수는 ETF 운용 과정에서 연간 기준으로 차감되는 비용 비율입니다."),
                ("추적배수", "추적배수는 ETF가 기초지수의 일일 수익률을 몇 배 또는 역방향으로 따라가도록 설계됐는지 나타냅니다."),
                ("복제방법", "복제방법은 ETF가 지수를 따라가기 위해 실물 자산 또는 파생상품을 사용하는 방식을 뜻합니다."),
                ("기초지수", "기초지수는 ETF가 추종 대상으로 삼는 시장 지표입니다."),
                ("과세유형", "ETF 과세는 상품 구조와 기초자산에 따라 달라질 수 있어 최신 세법과 상품 설명서를 확인해야 합니다."),
            ]
            topic, answer = topics[index % len(topics)]
            user = f"{row['short_name_kr']}을 살펴보기 전에 ETF의 {topic} 개념을 설명해줘."
            rows.append(make_no_call_case(split, action, user, answer, f"concept:{row['code']}:{topic}"))

        # [Action 6] 카탈로그 미지원 실시간/수익률 질의 거절 (No Tool)
        elif action == "unsupported":
            fields = ["오늘 가격", "최근 1년 수익률", "순자산총액", "변동성", "배당수익률"]
            field = fields[index % len(fields)]
            user = f"{row['short_name_kr']}의 {field}을 알려줘."
            answer = f"제공된 정적 ETF 카탈로그에는 {field} 정보가 없어 조회할 수 없습니다. 최신 시장 데이터 제공처를 확인하세요."
            rows.append(make_no_call_case(split, action, user, answer, f"unsupported:{row['code']}:{field}"))

        # [Action 7] 모호한 질의 재확인 요청 (No Tool)
        elif action == "clarification":
            token = row["short_name_kr"].split()[0]
            user = f"{token} 상품 정보 알려줘."
            answer = "정확한 ETF를 특정할 수 없습니다. 6자리 종목코드 또는 전체 ETF 약명을 알려주세요."
            rows.append(make_no_call_case(split, action, user, answer, f"clarify:{row['code']}"))

        # [Action 8] 일반 일상/수학 계산 대화 (No Tool)
        elif action == "general":
            split_offset = {"train": 0, "validation": 10_000, "test": 20_000}[split]
            a, b = 10 + split_offset + index, 3 + (index % 17)
            user = f"{a}와 {b}를 더하면 얼마야?"
            answer = f"{a}와 {b}의 합은 {a + b}입니다."
            rows.append(make_no_call_case(split, action, user, answer, f"general:add:{a}:{b}"))
        else:
            raise ValueError(action)
    return rows

# -----------------------------------------------------------------------------
# 5. Glaive 오픈소스 무작위 샘플링 매핑 함수
# -----------------------------------------------------------------------------
def select_glaive(split, kind, count):
    """Glaive 파싱 파이프라인에서 추출한 call / no_call 데이터셋 추출"""
    pool = glaive_pools[split][kind]
    if len(pool) < count:
        raise ValueError(f"Glaive {split}/{kind} 샘플 부족: {len(pool)} < {count}")
    selected = random.Random(SEED).sample(pool, count)
    return [{
        "source": "glaive",
        "split": split,
        "action": "glaive_tool" if kind == "call" else "no_tool",
        "group_id": hashlib.sha256(json.dumps(item["messages"], ensure_ascii=False).encode()).hexdigest(),
        "user": next(message["content"] for message in item["messages"] if message["role"] == "user"),
        "gold_arguments": None,
        "gold_result": None,
        "gold_result_hash": None,
        "messages": item["messages"],
        "tools": None,
    } for item in selected]

# -----------------------------------------------------------------------------
# 6. Split별 데이터 합성 및 해시 group_id 교차 검증
# -----------------------------------------------------------------------------
tool_case_splits = {}
for split in ["train", "validation", "test"]:
    total = split_total(CFG["tool_total"], split)
    counts = action_counts(total)
    cases = []
    for action in [
        "get_etf_details", "search_etfs", "compare_etfs", "aggregate_etfs_sql",
        "etf_concept", "unsupported", "clarification", "general",
    ]:
        cases.extend(synthesize_action(split, action, counts[action]))
    cases.extend(select_glaive(split, "call", counts["glaive_call"]))
    cases.extend(select_glaive(split, "no_call", counts["glaive_no_call"]))
    random.Random(SEED + PHRASE_OFFSET[split]).shuffle(cases)
    tool_case_splits[split] = cases

# Data Leakage 차단을 위한 group_id 비교 검증
group_sets = {split: {case["group_id"] for case in cases} for split, cases in tool_case_splits.items()}
assert group_sets["train"].isdisjoint(group_sets["validation"] | group_sets["test"])
assert group_sets["validation"].isdisjoint(group_sets["test"])
for split, cases in tool_case_splits.items():
    print(split, len(cases), Counter(case["action"] for case in cases))


**실행 결과 설명**

| split | 전체 | no-tool | 각 주요 tool |
|---|---:|---:|---|
| train | 2,100 | 840 | details/search/SQL 각 315, compare 210 |
| validation | 450 | 182 | details/search/SQL 각 67, compare 45 |
| test | 450 | 182 | details/search/SQL 각 67, compare 45 |

no-tool이 40%로 가장 큰 이유는 “도구를 쓰지 않아야 하는 능력”도 함께 학습하기 위해서입니다. Glaive tool 사례는 train 105개로 전체의 5%만 섞어 ETF 도메인 데이터가 중심이 되도록 했습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 합성된 Tool Calling 학습 데이터셋 샘플 구조 및 턴 출력
# -----------------------------------------------------------------------------
print("=== [1. Tool Call 필요 사례 샘플 (get_etf_details)] ===")
call_sample = next(case for case in tool_case_splits["train"] if case["action"] == "get_etf_details")
print(f"- User Query    : {call_sample['user']}")
print(f"- Gold Arguments: {call_sample['gold_arguments']}")
print(f"- Messages (Turn): {json.dumps(call_sample['messages'], ensure_ascii=False, indent=2)}")

print("\n=== [2. No-Tool (도구 미사용) 일반 대화 사례 샘플] ===")
no_call_sample = next(case for case in tool_case_splits["train"] if case["action"] == "no_tool")
print(f"- Category     : {no_call_sample['category']}")
print(f"- User Query   : {no_call_sample['user']}")
print(f"- Messages     : {json.dumps(no_call_sample['messages'], ensure_ascii=False, indent=2)}")


### **2-7. Tool Calling은 Instruct 모델에서 시작하기**

Part 2는 이미 대화와 지시 수행 능력이 있는 `Qwen3-4B-Instruct-2507`을 사용합니다. 도메인별 tool schema와 호출 규칙을 추가로 익히는 목적이기 때문입니다.

LoRA rank를 `r=32`로 설정해 Part 1보다 큰 adapter 용량을 사용합니다. rank가 두 배라고 성능이 자동으로 두 배가 되는 것은 아니며, 데이터 규모·난이도·과적합을 함께 봐야 합니다.


In [ ]:
# -----------------------------------------------------------------------------
# Part 2: Tool Calling 전용 Instruct 모델 로드 및 LoRA (r=32) 어댑터 부착
# -----------------------------------------------------------------------------
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

# Tool Calling 용도로 지시 따르기(Instruction-following) 능력이 기본 학습된 Instruct 모델 사용
TC_MODEL = "unsloth/Qwen3-4B-Instruct-2507"

# 1. 4-bit NF4 양자화 방식으로 Instruct 모델 및 토커나이저 로드
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=TC_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,        # VRAM 절약을 위한 4-bit 양자화
    load_in_8bit=False,
    full_finetuning=False,    # LoRA 어댑터 미세조정 사용
    token=HF_TOKEN,
)
# Qwen3 Instruct 포맷 대화 챗 템플릿 적용
tokenizer = get_chat_template(tokenizer, chat_template="qwen3-instruct")

# 2. LoRA 랭크 r=32 (Part 1 대비 2배 용량) 적용 및 전체 7개 선형 모듈 타겟팅
model = FastLanguageModel.get_peft_model(
    model,
    r=32,  # 복잡한 JSON Schema 및 Tool Argument 생성을 위해 표현 용량(Rank) 확장
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention Layer
        "gate_proj", "up_proj", "down_proj",     # MLP / FFN Layer
    ],
    lora_alpha=32,                        # LoRA 스케일링 계수 (r=32과 1:1)
    lora_dropout=0,                       # Unsloth 고속 커널을 위해 0 권장
    bias="none",                           # 바이어스 파라미터 제외
    use_gradient_checkpointing="unsloth",  # 메모리 절약 체크포인팅
    random_state=SEED,                    # 시드 고정
)
print(f"✅ {TC_MODEL} + LoRA r=32")


**실행 결과 설명**

4-bit Instruct 모델이 정상 로드되고 LoRA `r=32`가 붙었습니다. padding token 자동 지정은 Part 1과 같은 안내입니다. 이후 trainer가 표시한 학습 가능 파라미터는 약 66.1M으로 Part 1의 약 두 배입니다.


### **2-8. Tool Calling 평가 퍼널**

한 번의 호출은 여러 관문을 차례로 통과해야 합니다.

`호출 여부 → JSON 파싱 → Schema 유효성 → 도구 이름 → 인자 → 실행 성공 → 실행 결과 정답`

- **Precision** $=TP/(TP+FP)$: 호출한 것 중 호출해야 했던 비율
- **Recall** $=TP/(TP+FN)$: 호출해야 한 것 중 실제 호출한 비율
- **F1** $=2PR/(P+R)$: precision과 recall의 조화평균
- `over_call_rate`: 도구가 필요 없는 요청에서 잘못 호출한 비율
- `argument_exact_rate`: 정답 인자와 JSON 값이 정확히 같은 비율
- `execution_success_rate`: 실행 자체가 오류 없이 끝난 비율
- `execution_accuracy`: 실행 결과까지 gold와 의미적으로 같은 비율

Template은 학습 합성기와 비슷한 문장, Challenge는 새 표현·복합 조건·모호성·미지원 요청을 사용합니다. 두 결과를 분리해야 암기와 일반화를 구분할 수 있습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Qwen 챗 템플릿 기반 ETF Tool 추론 생성 함수
# -----------------------------------------------------------------------------
IM_END_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")
EOS_IDS = [tokenizer.eos_token_id, IM_END_ID]

def generate_with_etf_tools(user_text, messages=None, max_new_tokens=256):
    """ETF_TOOLS 스키마를 포함한 시스템 프롬프트 적용 및 Greedy 추론 생성"""
    messages = messages or [{"role": "user", "content": user_text}]
    inputs = tokenizer.apply_chat_template(
        messages,
        tools=ETF_TOOLS,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=EOS_IDS,
        )
    generated = output[0][inputs["input_ids"].shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# -----------------------------------------------------------------------------
# 2. 모델 출력 응답에서 <tool_call> JSON 태그 추출 및 인자 검증
# -----------------------------------------------------------------------------
def extract_tool_call(response):
    """생성 텍스트에서 <tool_call> 태그 내의 JSON 블록 추출 및 파싱"""
    match = re.search(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", response, re.S)
    if not match:
        return None
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None

def validate_tool_arguments(name, arguments):
    """도구별 인자의 무결성, 컬럼 화이트리스트, 정렬/범위/SQL AST 문법 검증"""
    if name not in TOOL_NAMES or not isinstance(arguments, dict):
        return False
    try:
        if name == "get_etf_details":
            return isinstance(arguments.get("identifier"), str) and bool(checked_fields(arguments.get("fields")))
        if name == "compare_etfs":
            identifiers = arguments.get("identifiers")
            return isinstance(identifiers, list) and 2 <= len(identifiers) <= 5 and bool(checked_fields(arguments.get("fields")))
        if name == "search_etfs":
            checked_fields(arguments.get("return_fields"))
            if arguments.get("sort_by") not in SORTABLE_COLUMNS or arguments.get("sort_order") not in {"asc", "desc"}:
                return False
            if not 1 <= int(arguments.get("limit", 0)) <= 20:
                return False
            filter_keys = {"keyword", "manager", "market", "asset_class", "tracking_multiple", "min_fee", "max_fee"}
            return any(arguments.get(key) not in (None, "") for key in filter_keys)
        if name == "aggregate_etfs_sql":
            validate_aggregate_sql(arguments.get("sql", ""))
            return True
    except (KeyError, TypeError, ValueError, sqlglot.errors.ParseError):
        return False
    return False

# -----------------------------------------------------------------------------
# 3. 결과 의미론적 동치성(Denotation Equivalence) 및 nDCG / Macro F1 지표 계산
# -----------------------------------------------------------------------------
def canonical_result(result):
    """결과 튜플 및 스칼라 정규화"""
    if not result:
        return None
    return {
        "columns": result["columns"],
        "rows": [[normalize_scalar(value) for value in row] for row in result["rows"]],
    }

def denotation_equivalent(action, expected, predicted):
    """SQL 쿼리 형태가 달라도 DB 실행 결과가 실질적으로 완전히 동등한지 검증"""
    expected = canonical_result(expected)
    predicted = canonical_result(predicted)
    if expected is None or predicted is None:
        return expected == predicted
    if action == "aggregate_etfs_sql":
        # alias 이름은 달라도 집계 셀 값과 행 순서가 같으면 실행 의미가 같다고 봄
        return expected["rows"] == predicted["rows"]
    return expected == predicted

def macro_f1(truth, prediction, labels):
    """클래스별 F1-score의 단순 평균(Macro F1) 계산"""
    scores = []
    for label in labels:
        tp = sum(t == label and p == label for t, p in zip(truth, prediction))
        fp = sum(t != label and p == label for t, p in zip(truth, prediction))
        fn = sum(t == label and p != label for t, p in zip(truth, prediction))
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        scores.append(2 * precision * recall / (precision + recall) if precision + recall else 0.0)
    return float(np.mean(scores))

def search_ndcg(expected, predicted):
    """검색 결과 순위의 정밀도 평가 (nDCG@K)"""
    if not expected or "code" not in expected["columns"]:
        return float("nan")
    code_index = expected["columns"].index("code")
    relevant = {row[code_index] for row in expected["rows"]}
    if not predicted or "code" not in predicted["columns"]:
        return 0.0
    predicted_index = predicted["columns"].index("code")
    ranking = [row[predicted_index] for row in predicted["rows"]]
    dcg = sum((1.0 if code in relevant else 0.0) / math.log2(rank + 2) for rank, code in enumerate(ranking))
    ideal = sum(1.0 / math.log2(rank + 2) for rank in range(min(len(relevant), len(ranking))))
    return dcg / ideal if ideal else 0.0

def stratified_eval_cases(cases, limit):
    """액션별 비율 균등 샘플링"""
    by_action = defaultdict(list)
    for case in cases:
        if case["source"] != "glaive":
            by_action[case["action"]].append(case)
    per_action = max(1, limit // len(by_action))
    selected = []
    for action, rows in sorted(by_action.items()):
        selected.extend(rows[:per_action])
    return selected[:limit]

tool_eval_cases = stratified_eval_cases(tool_case_splits["test"], CFG["tool_eval_n"])

# -----------------------------------------------------------------------------
# 4. 미학습 문구/복합 조건을 위한 Challenge held-out 벤치마크 생성
# -----------------------------------------------------------------------------
def build_tool_challenge_cases(limit):
    """학습 합성기에 사용되지 않은 새로운 표현, 고난도 복합 조건, 거절 질의로 구성된 챌린지 셋"""
    frame = etf_df[etf_df["split"] == "test"].reset_index(drop=True)
    per_call_action = max(2, limit // 8)
    cases = []
    for index in range(per_call_action):
        row = frame.iloc[index % len(frame)]

        get_fields = ["manager", "total_fee", "tax_type"]
        get_case = make_call_case(
            "test", "get_etf_details",
            f"종목코드 {row['code']}에서 상품명은 빼고 운용 주체, 연간 비용, 세금 분류만 뽑아 줘.",
            {"identifier": row["code"], "fields": get_fields},
            f"challenge:get:{row['code']}", index,
        )
        cases.append(get_case)

        search_args = {
            "manager": row["manager"], "asset_class": row["asset_class"],
            "max_fee": float(row["total_fee"]),
            "return_fields": ["short_name_kr", "manager", "total_fee"],
            "sort_by": "total_fee", "sort_order": "asc", "limit": 3,
        }
        search_case = make_call_case(
            "test", "search_etfs",
            f"운용사는 {row['manager']}, 분류는 {row['asset_class']}, 비용 상한은 {row['total_fee']:.3f}%야. 조건을 모두 만족하는 상품을 싼 것부터 셋만.",
            search_args, f"challenge:search:{row['code']}", index,
        )
        cases.append(search_case)

        same_asset = frame[frame["asset_class"] == row["asset_class"]]
        compare_pool = same_asset if len(same_asset) >= 2 else frame
        compare_n = min(2 + index % 2, len(compare_pool))
        identifiers = compare_pool.sample(compare_n, random_state=SEED + 500 + index)["code"].tolist()
        compare_fields = ["short_name_kr", "base_index", "total_fee", "tracking_multiple"]
        compare_joined = "; ".join(identifiers)
        compare_group = ":".join(identifiers)
        compare_case = make_call_case(
            "test", "compare_etfs",
            f"{compare_joined} — 이 코드들을 지수, 비용, 추적 방식까지 한 표로 대조해 줘.",
            {"identifiers": identifiers, "fields": compare_fields},
            f"challenge:compare:{compare_group}", index,
        )
        cases.append(compare_case)

        if index % 2 == 0:
            sql = f"SELECT COUNT(*) AS etf_count FROM etfs WHERE manager = {sql_quote(row['manager'])} AND asset_class = {sql_quote(row['asset_class'])}"
            aggregate_user = f"{row['manager']}이 운용하는 {row['asset_class']} ETF가 총 몇 종목인지 상세 목록 말고 숫자만 계산해 줘."
        else:
            sql = f"SELECT AVG(total_fee) AS average_fee FROM etfs WHERE market = {sql_quote(row['market'])} AND tracking_multiple = {sql_quote(row['tracking_multiple'])}"
            aggregate_user = f"{row['market']} 시장의 {row['tracking_multiple']} 상품 전체에 대한 평균 비용률을 집계해 줘."
        aggregate_case = make_call_case(
            "test", "aggregate_etfs_sql", aggregate_user, {"sql": sql},
            f"challenge:sql:{index}:{row['code']}", index,
        )
        cases.append(aggregate_case)

    no_call_index = 0
    while len(cases) < limit:
        row = frame.iloc[no_call_index % len(frame)]
        mode = no_call_index % 4
        if mode == 0:
            user, answer, category = (
                f"{row['short_name_kr']}의 지금 체결가와 오늘 거래량을 바로 알려줘.",
                "정적 ETF 카탈로그에는 실시간 체결가와 거래량이 없습니다. 최신 시장 데이터 제공처가 필요합니다.",
                "challenge_unsupported",
            )
        elif mode == 1:
            first_token = row["short_name_kr"].split()[0]
            user, answer, category = (
                f"{first_token} 그 상품, 정보 좀.",
                "상품을 특정할 수 없습니다. 6자리 종목코드나 전체 ETF 약명을 알려주세요.",
                "challenge_clarification",
            )
        elif mode == 2:
            user, answer, category = (
                "특정 상품 조회 없이 ETF의 실물복제와 합성복제 차이만 설명해 줘.",
                "실물복제는 구성 자산을 직접 보유하고, 합성복제는 스왑 등 파생계약으로 지수 수익률을 추종합니다.",
                "challenge_concept",
            )
        else:
            a, b = 701 + no_call_index, 29
            user, answer, category = (f"도구는 쓰지 말고 {a}에서 {b}를 빼 줘.", f"{a - b}입니다.", "challenge_general")
        cases.append(make_no_call_case("test", category, user, answer, f"challenge:no_call:{no_call_index}"))
        no_call_index += 1

    for case in cases:
        case["source"] = "challenge"
    return cases[:limit]

tool_challenge_cases = build_tool_challenge_cases(CFG["tool_challenge_n"])
print("Challenge distribution:", Counter(case["action"] for case in tool_challenge_cases))

# -----------------------------------------------------------------------------
# 5. 다단계 퍼널 평가기 (Call Precision/Recall, Schema Valid, Argument/Denotation Accuracy)
# -----------------------------------------------------------------------------
def evaluate_tool_model(cases, label):
    """Tool Calling 모델의 호출, 스키마 유효성, 인자 일치율, 백엔드 실행 성공률 및 결과 동치성 측정"""
    records = []
    FastLanguageModel.for_inference(model)
    for case in cases:
        response = generate_with_etf_tools(case["user"])
        parsed = extract_tool_call(response)
        must_call = case["action"] != "no_tool"
        predicted_name = parsed.get("name") if parsed else "no_tool"
        arguments = parsed.get("arguments", {}) if parsed else {}
        schema_valid = bool(parsed) and validate_tool_arguments(predicted_name, arguments)
        tool_correct = predicted_name == case["action"]
        args_correct = tool_correct and arguments == case["gold_arguments"]
        predicted_result = None
        execution_success = False
        if schema_valid:
            try:
                predicted_result = execute_tool_call(predicted_name, arguments, DB_PATHS["test"])
                execution_success = True
            except Exception:
                execution_success = False
        denotation = execution_success and denotation_equivalent(case["action"], case["gold_result"], predicted_result)
        records.append({
            "truth": case["action"],
            "prediction": predicted_name,
            "must_call": must_call,
            "called": parsed is not None,
            "json_valid": parsed is not None,
            "schema_valid": schema_valid,
            "tool_correct": tool_correct,
            "args_correct": args_correct,
            "execution_success": execution_success,
            "denotation": denotation,
            "search_ndcg": search_ndcg(case["gold_result"], predicted_result) if case["action"] == "search_etfs" else np.nan,
            "response": response,
        })
    frame = pd.DataFrame(records)
    truth_call = frame["must_call"].astype(bool)
    pred_call = frame["called"].astype(bool)
    tp = int((truth_call & pred_call).sum())
    fp = int((~truth_call & pred_call).sum())
    fn = int((truth_call & ~pred_call).sum())
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    summary = {
        "label": label,
        "n": len(frame),
        "call_precision": precision,
        "call_recall": recall,
        "call_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "action_macro_f1": macro_f1(frame["truth"], frame["prediction"], ["get_etf_details", "search_etfs", "compare_etfs", "aggregate_etfs_sql", "no_tool"]),
        "over_call_rate": fp / max(1, int((~truth_call).sum())),
        "under_call_rate": fn / max(1, int(truth_call.sum())),
        "json_parse_rate": float(frame.loc[truth_call, "json_valid"].mean()),
        "schema_valid_rate": float(frame.loc[truth_call, "schema_valid"].mean()),
        "tool_name_accuracy": float(frame.loc[truth_call, "tool_correct"].mean()),
        "argument_exact_rate": float(frame.loc[truth_call, "args_correct"].mean()),
        "execution_success_rate": float(frame.loc[truth_call, "execution_success"].mean()),
        "execution_accuracy": float(frame.loc[truth_call, "denotation"].mean()),
        "search_ndcg": float(frame["search_ndcg"].mean()),
    }
    print(pd.Series(summary).to_frame().T)
    return summary, frame

# -----------------------------------------------------------------------------
# 6. 파인튜닝 전(Before FT) Template 셋 및 Challenge 셋 고정 평가 실행
# -----------------------------------------------------------------------------
tool_before_summary, tool_before_rows = evaluate_tool_model(tool_eval_cases, "Before-Template")
tool_before_challenge_summary, tool_before_challenge_rows = evaluate_tool_model(tool_challenge_cases, "Before-Challenge")


**학습 전 baseline 결과 설명**

- Template: call F1 0.9697이지만 tool name 0.8333, argument exact 0.3854, execution accuracy 0.4479입니다.
- Challenge: correct tool 0.70, exact arguments 0.25, execution accuracy 0.35, over-call 0.25입니다.
- Challenge `execution_success=1.0`은 생성된 유효 호출이 오류 없이 실행됐다는 뜻이지, 정답 호출이라는 뜻이 아닙니다.

즉 Instruct 모델이 JSON과 호출 형태는 만들 수 있지만 ETF 규칙에 맞는 도구·인자 선택은 아직 부족합니다.


### **2-9. Tool schema를 포함한 ChatML과 학습 설정**

각 대화에 해당 tool schema를 함께 넣어 모델이 “사용 가능한 함수와 인자 계약”을 보면서 응답하도록 합니다. Glaive 사례는 원래 system schema를 보존합니다.

- device batch 2 × accumulation 8 = **유효 배치 16**
- 50 step마다 validation과 checkpoint 저장
- assistant 영역만 response-only loss 적용
- `learned_tokens > 0` assertion으로 마스킹 실수 탐지

`sample_chars`는 문자 수이지 토큰 수가 아닙니다. 실제 2048-token 제한은 tokenizer가 판단합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. ETF Tool Schema를 포함한 ChatML 대화 직렬화(Serialization) 함수
# -----------------------------------------------------------------------------
from trl import SFTConfig, SFTTrainer

def render_tool_case(case):
    """Tool 정의(tools) 스키마를 tokenizer에 전달하여 <tool_call> 직렬화 대화 생성"""
    kwargs = {}
    if case["tools"] is not None:
        kwargs["tools"] = case["tools"]
    return tokenizer.apply_chat_template(
        case["messages"],
        tokenize=False,
        add_generation_prompt=False,
        **kwargs,
    )

# train/validation 데이터 세트 텍스트 포맷 직렬화
tool_text_splits = {}
for split in ["train", "validation"]:
    texts = [render_tool_case(case) for case in tool_case_splits[split]]
    tool_text_splits[split] = Dataset.from_list([{"text": text} for text in texts])
    print(split, len(texts), "sample_chars=", len(texts[0]))

# -----------------------------------------------------------------------------
# 2. Tool Calling 학습용 SFTTrainer 및 SFTConfig 파라미터 설정
# -----------------------------------------------------------------------------
trainer_tc = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=tool_text_splits["train"],
    eval_dataset=tool_text_splits["validation"],
    args=SFTConfig(
        output_dir=str(WORK_DIR / "tool_calling_checkpoints"),  # 체크포인트 저장 경로
        num_train_epochs=1,                                    # 1 Epoch 학습
        per_device_train_batch_size=2,                         # 디바이스 당 배치 수 (2)
        gradient_accumulation_steps=8,                         # 그래디언트 누적 단계 (8) -> 유효 배치 = 2x8 = 16
        learning_rate=2e-4,                                    # 학습률 (2e-4)
        warmup_ratio=0.10,                                     # 웜업 비율 (10%)
        lr_scheduler_type="cosine",                             # 코사인 학습률 스케줄러
        optim="adamw_8bit",                                    # 8-bit AdamW 최적화 (VRAM 절감)
        bf16=True,                                             # BF16 연산 가속
        max_seq_length=MAX_SEQ_LENGTH,                         # 최대 토큰 시퀀스 길이 (2048)
        logging_steps=10,                                      # 10 스텝마다 로그
        eval_strategy="steps",                                 # 50 스텝마다 검증 평가 수행
        eval_steps=50,
        save_strategy="steps",                                 # 50 스텝마다 체크포인트 저장
        save_steps=50,
        save_total_limit=1,                                    # 체크포인트 1개만 저장
        load_best_model_at_end=True,                           # 가장 우수한 eval_loss 모델 복원
        metric_for_best_model="eval_loss",                     # 최고 모델 평가 지표
        greater_is_better=False,                               # eval_loss는 작을수록 양호
        report_to="none",                                      # 로깅 비활성화
        seed=SEED,                                             # 시드 고정
        dataset_text_field="text",                             # 입력 필드 지정
    ),
)

# -----------------------------------------------------------------------------
# 3. Response-only Loss 마스킹 적용 (Assistant 도구 호출 및 답변 영역만 학습)
# -----------------------------------------------------------------------------
trainer_tc = train_on_responses_only(
    trainer_tc,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
# Assistant 학습 영역 마스킹 검증 (learned_tokens > 0)
learned = sum(label != -100 for label in trainer_tc.train_dataset[0]["labels"])
assert learned > 0
print(f"✅ Tool response-only labels: learned_tokens={learned}")


**실행 결과 설명**

train 2,100개와 validation 450개가 ChatML 텍스트로 변환됐습니다. 첫 예시는 약 3천 문자이며, tokenizer 처리와 response-only 마스킹이 완료됐습니다.

`learned_tokens=36`은 첫 사례에서 assistant가 생성해야 할 tool call 또는 답변 토큰 36개가 loss 대상이라는 뜻입니다. 0이면 marker가 template과 맞지 않아 학습이 무효가 되므로 assertion이 중요합니다.


### **2-10. Tool Calling 학습 후 같은 테스트셋으로 재평가**

학습 전과 후에 동일한 Template/Challenge 사례와 greedy decoding을 사용합니다. 출력 표에서는 한 지표만 보지 말고 다음 순서로 읽습니다.

1. 도구가 필요한지 판단했는가?
2. 올바른 도구를 골랐는가?
3. 인자가 정확하고 schema에 맞는가?
4. 실행 결과가 실제 정답과 같은가?

train loss가 작아도 Challenge의 `execution_accuracy`가 낮다면 새로운 표현에서 인자를 만드는 능력이 아직 부족한 것입니다.


In [ ]:
# -----------------------------------------------------------------------------
# Part 2: ETF Tool Calling 파인튜닝 학습 실행
# -----------------------------------------------------------------------------
print("🚀 Part 2 ETF Tool Calling FT 시작")
FastLanguageModel.for_training(model)  # 학습 전용 최적화 연산 커널로 전환
tc_stats = trainer_tc.train()         # SFT 학습 진행 (132 Steps / 1 Epoch)
tc_validation = trainer_tc.evaluate()   # 검증 오차(eval_loss) 측정
print(f"train_loss={tc_stats.training_loss:.4f} | eval_loss={tc_validation['eval_loss']:.4f}")

# -----------------------------------------------------------------------------
# 학습 후(After FT) Template 셋 및 Challenge 셋 고정 평가 실행
# -----------------------------------------------------------------------------
tool_after_summary, tool_after_rows = evaluate_tool_model(tool_eval_cases, "After-Template")
tool_after_challenge_summary, tool_after_challenge_rows = evaluate_tool_model(tool_challenge_cases, "After-Challenge")

# -----------------------------------------------------------------------------
# 학습 전/후 (Before vs After) Template 및 Challenge 종합 지표 비교 출력
# -----------------------------------------------------------------------------
comparison = pd.DataFrame([
    tool_before_summary, tool_after_summary,
    tool_before_challenge_summary, tool_after_challenge_summary,
]).set_index("label")
print(comparison.drop(columns="n").T.round(4))

# [성능 포화 및 개선 여지 판별 안내]
if tool_after_summary["execution_accuracy"] >= 0.99 and tool_after_challenge_summary["execution_accuracy"] < 0.99:
    print("ℹ️ template benchmark는 포화됐지만 challenge 성능에는 개선 여지가 있습니다.")
if tool_after_challenge_summary["execution_accuracy"] >= 0.99:
    print("⚠️ challenge도 포화됐습니다. 실제 사용자 로그 기반 adversarial case를 추가하세요.")

# -----------------------------------------------------------------------------
# 평가 결과 JSON 파일 및 Tool Calling LoRA 어댑터/토커나이저 저장
# -----------------------------------------------------------------------------
tool_metrics_path = WORK_DIR / "tool_calling_eval_metrics.json"
with tool_metrics_path.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "template": {"before": tool_before_summary, "after": tool_after_summary},
            "challenge": {"before": tool_before_challenge_summary, "after": tool_after_challenge_summary},
        },
        file,
        ensure_ascii=False,
        indent=2,
        default=float,
    )

tc_save_path = WORK_DIR / "qwen3_etf_tool_calling_lora"
model.save_pretrained(str(tc_save_path))      # Tool Calling 전용 LoRA 어댑터 저장
tokenizer.save_pretrained(str(tc_save_path))  # 토커나이저 저장
if PUSH_TO_HUB:
    repo_id = f"{HF_NAMESPACE}/qwen3-4b-etf-tool-calling-lora"
    model.push_to_hub(repo_id, token=HF_TOKEN, private=HF_PRIVATE)
    tokenizer.push_to_hub(repo_id, token=HF_TOKEN, private=HF_PRIVATE)
print(f"✅ Tool LoRA={tc_save_path} | metrics={tool_metrics_path}")


**실행 결과 설명 — 개선과 남은 오류를 함께 보기**

- 2,100개, 1 epoch, 132 step, 유효 배치 16
- train loss 0.2403, eval loss 0.0869
- Template는 모든 지표 1.0으로 포화
- Challenge correct tool `0.70→1.00`
- Challenge argument exact `0.25→0.75`
- Challenge execution accuracy `0.35→0.75`
- Challenge over-call `0.25→0.10`

must-call 20개 중 실행 의미가 틀린 사례가 5개 남고, no-tool 20개 중 2개는 여전히 over-call입니다. Template 1.0은 합성기와 비슷한 문구를 잘 학습했다는 뜻이며 실제 사용자 표현 전체에 대한 완성 판정이 아닙니다.


### **2-11. Tool Calling의 실제 4단계 왕복**

```text
[1] User 요청
  ↓
[2] Assistant가 <tool_call> 생성
  ↓ 검증 후 실행
[3] Tool이 구조화된 JSON 결과 반환
  ↓
[4] Assistant가 사용자용 자연어 답변 생성
```

중요한 점은 [2]와 [4]가 다르다는 것입니다. 모델은 먼저 호출을 제안하고, 애플리케이션이 검증·실행한 신뢰 가능한 결과를 다시 모델에 넣어 최종 답변을 만듭니다.


In [ ]:
# -----------------------------------------------------------------------------
# 실제 SQLite DB 연동 기반 Tool Calling 4단계 멀티턴(Multi-turn) 대화 데모
# -----------------------------------------------------------------------------
# 1. 테스트 세트에서 search_etfs 도구 호출 사례 하나 선택
demo_case = next(case for case in tool_eval_cases if case["action"] == "search_etfs")

# [Step 1] 사용자 질문 생성 및 모델의 1차 도구 호출 제안(<tool_call>) 유도
first_response = generate_with_etf_tools(demo_case["user"])
predicted_call = extract_tool_call(first_response)
print("[1] User:", demo_case["user"])
print("[2] Assistant:", first_response)

# [Step 2] 추출된 도구 인자 보안 검증 후, 실제 full SQLite DB 조회를 백엔드에서 실행
if predicted_call and validate_tool_arguments(predicted_call["name"], predicted_call["arguments"]):
    tool_result = execute_tool_call(predicted_call["name"], predicted_call["arguments"], DB_PATHS["full"])
    print("[3] Tool:", json.dumps(tool_result, ensure_ascii=False)[:1_000])
    
    # [Step 3] DB 조회 결과를 멀티턴 대화 히스토리(role: tool)로 재조합
    loop_messages = [
        {"role": "user", "content": demo_case["user"]},
        {"role": "assistant", "content": first_response},
        {"role": "tool", "content": json.dumps(tool_result, ensure_ascii=False)},
    ]
    
    # [Step 4] DB 조회 결과를 바탕으로 사용자에게 최종 자연어 답변(Grounded Response) 생성
    final_response = generate_with_etf_tools(demo_case["user"], messages=loop_messages)
    print("[4] Assistant:", final_response)
else:
    print("❌ 유효한 tool call을 생성하지 못했습니다.")


**실행 결과 설명**

모델은 `search_etfs`와 keyword·반환 열·보수 오름차순·limit 5를 포함한 유효 JSON을 만들었습니다. 애플리케이션이 SQLite에서 ETF 한 건을 찾았고, 모델은 tool 결과를 근거로 최종 자연어 답변을 생성했습니다.

이 한 사례는 end-to-end 연결이 작동한다는 smoke test입니다. Challenge 40건의 집계 지표를 대신하는 일반화 증거는 아닙니다.


### Part 2 실행 결과 해석 — 개선 확인, 일반화 주의

- Glaive 20,000건 중 18,791건(93.96%)을 변환했고 1,209건을 거절했습니다. 학습 2,100건은 ETF·Glaive 호출과 no-tool 요청을 혼합했습니다.
- Template benchmark는 After가 전 지표 1.0이지만 합성기와 동일한 문구 분포이므로 포화 결과입니다.
- 별도 challenge 40건에서 correct tool 0.70→1.00, exact arguments 0.25→0.75, semantic execution 0.35→0.75, over-call 0.25→0.10으로 개선됐습니다.
- execution success는 Before도 1.0이었습니다. 실행 가능하다는 사실만으로 정답 호출이라는 뜻은 아니며, After도 must-call 20건 중 5건은 의미적으로 틀렸고 no-tool 20건 중 2건은 over-call했습니다.
- search nDCG 1.0은 검색 사례가 5건뿐이고 기존 ideal denominator가 짧은 예측을 과대평가할 수 있어 보수적으로 해석해야 합니다.

교육용 QUICK 결과로는 도구 호출 개선을 보여주지만, 실제 배포 일반화나 완전한 인자 정확성을 의미하지 않습니다.


---

# Part 3: Embedding FT — ETF 1차 검색기

Tool Calling이 “무슨 동작을 실행할지”를 학습했다면, Embedding FT는 “어떤 문서를 후보로 가져올지”를 학습합니다.

- `corpus`: 실제 ETF 필드를 합친 canonical 문서
- `queries`: ETF 정보·보수·지수·속성 질의
- `qrels`: entity query의 단일 정답과 attribute challenge의 multi-positive 정답 집합
- split: Part 2와 동일한 ETF entity split
- primary metric: MRR@10 · nDCG@10 · Recall@10

Pair AP와 Spearman은 분리 능력의 보조 진단으로만 사용합니다. ETF 이름·코드가 들어간 entity 점수가 포화되어도 속성만 주어진 challenge 점수는 별도로 해석합니다.


### **3-1. BGE-M3 Embedding 모델 준비**

Embedding 모델은 문장을 고정 길이 숫자 벡터로 바꿉니다. 의미가 비슷한 query와 document가 벡터 공간에서 가까워지도록 학습합니다.

$$\text{cosine}(q,d)=\frac{q\cdot d}{\lVert q\rVert\lVert d\rVert}$$

`normalize_embeddings=True`이면 벡터 길이가 1이므로 cosine similarity가 단순 내적 `q @ d`와 같습니다.

BGE-M3의 기본 차원은 1024이고, 이 실습은 Matryoshka 학습을 위해 1024·512·128차원 성능을 함께 측정합니다. 차원이 작으면 저장 공간과 검색 계산량은 줄지만 정보 손실 가능성이 있습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Part 2 LLM 메모리 해제 및 가비지 컬렉션 (VRAM 정돈)
# -----------------------------------------------------------------------------
del model, trainer_tc, tokenizer
gc.collect()
torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# 2. SentenceTransformers 임베딩 모델 / 트레이너 / 평가 모듈 임포트
# -----------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer
from sentence_transformers.evaluation import (
    BinaryClassificationEvaluator,
    EmbeddingSimilarityEvaluator,
    InformationRetrievalEvaluator,
)
from sentence_transformers.losses import MatryoshkaLoss, MultipleNegativesRankingLoss
from sentence_transformers.training_args import (
    BatchSamplers,
    MultiDatasetBatchSamplers,
    SentenceTransformerTrainingArguments,
)

# -----------------------------------------------------------------------------
# 3. BAAI/bge-m3 다국어 dense 임베딩 모델 로드 (bfloat16 연산 가속)
# -----------------------------------------------------------------------------
EMBEDDING_MODEL = "BAAI/bge-m3"
emb_model = SentenceTransformer(
    EMBEDDING_MODEL,
    model_kwargs={"dtype": torch.bfloat16},  # 메모리 절약 및 Ampere GPU 가속용 bfloat16 설정
)

# Matryoshka 학습 기법 평가를 위한 3가지 슬라이싱 임베딩 차원 크기 정의 [1024, 512, 128]
EMBEDDING_DIMS = [1024, 512, 128]
print(f"✅ {EMBEDDING_MODEL} | dim={emb_model.get_sentence_embedding_dimension()}")


**실행 결과 설명**

Part 2 LLM을 해제한 뒤 BGE-M3가 로드됐고 기본 embedding 차원 1024가 확인됐습니다. 이 시점의 모델은 아직 ETF 데이터로 학습하지 않은 **pretrained baseline**입니다.


### **3-2. corpus·query·qrels 만들기**

- `corpus`: 검색 대상 ETF 문서 300개
- `queries`: 사용자가 입력할 검색 문장
- `qrels`: 각 query와 관련 있는 정답 문서 ID 집합

**Entity query**는 상품명이나 코드가 들어 있어 정답 하나를 찾기 쉽습니다. **Attribute challenge**는 이름 없이 `운용사+자산군`처럼 속성만 제시하고 여러 ETF가 정답이므로 실제 검색 난도가 높습니다.

평균 정답 수가 약 14개라는 출력은 attribute query 하나에 관련 ETF가 평균 14개 있다는 뜻입니다. 따라서 첫 정답 하나의 순위뿐 아니라 관련 문서를 얼마나 많이 회수했는지도 평가해야 합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. ETF 카탈로그 텍스트 렌더링 (make_document) 및 Corpus / Query / Qrels 생성
# -----------------------------------------------------------------------------
def make_document(row):
    """ETF 개별 종목 속성을 검색용 표준 단락 서술 텍스트(Corpus Document)로 변환"""
    return (
        f"ETF 종목코드 {row['code']}, 상품명 {row['short_name_kr']}. "
        f"운용사 {row['manager']}, 기초시장 {row['market']}, 기초자산 {row['asset_class']}. "
        f"기초지수 {row['base_index']}, 추적배수 {row['tracking_multiple']}, "
        f"복제방법 {row['replication_method']}, 총보수 연 {row['total_fee']:.6f}%, "
        f"상장일 {row['listing_date']}, 과세유형 {row['tax_type']}."
    )

# Corpus(문서 고유 DB), Query(질의), Qrels(질의-정답 문서 매핑) 사전 구축
corpus = {row["code"]: make_document(row) for _, row in etf_df.iterrows()}
ir_queries = {split: {} for split in ["train", "validation", "test"]}
ir_qrels = {split: {} for split in ["train", "validation", "test"]}
ir_metadata = {}

# 종목명/코드가 명시되어 단일 정답(Single Positive)을 갖는 Entity Query 증강
for _, row in etf_df.iterrows():
    variants = [
        ("info", f"{row['short_name_kr']} ETF 정보를 알려줘"),
        ("fee", f"{row['short_name_kr']}의 총보수는 얼마야?"),
        ("index", f"{row['short_name_kr']}이 추종하는 기초지수는?"),
        ("attribute", f"{row['manager']}의 {row['asset_class']} 상품 {row['short_name_kr']}을 찾아줘"),
    ][:CFG["ir_query_variants"]]
    for intent, text in variants:
        query_id = f"{row['code']}:{intent}"
        split = row["split"]
        ir_queries[split][query_id] = text
        ir_qrels[split][query_id] = {row["code"]}
        ir_metadata[query_id] = {"doc_id": row["code"], "intent": intent, "split": split}

assert set().union(*[set(rows) for rows in ir_queries.values()]) == set(ir_metadata)
assert all(ir_qrels[split][qid] for split in ir_qrels for qid in ir_qrels[split])
print("corpus=", len(corpus), "queries=", {split: len(rows) for split, rows in ir_queries.items()})

# -----------------------------------------------------------------------------
# 2. 종목명 없이 속성 조합만으로 다중 정답(Multi-Positive)을 검색하는 Challenge 생성
# -----------------------------------------------------------------------------
ATTRIBUTE_CHALLENGE_SPECS = [
    (("asset_class", "market"), "{market} 시장에 속한 {asset_class} ETF를 찾아줘"),
    (("manager", "asset_class"), "{manager}이 운용하는 {asset_class} ETF 목록이 필요해"),
    (("tracking_multiple", "market"), "{market} 시장의 {tracking_multiple} 추적 ETF를 검색해줘"),
    (("replication_method", "asset_class"), "{replication_method} 방식으로 복제하는 {asset_class} ETF는?"),
    (("tax_type", "asset_class"), "{tax_type} 과세 유형인 {asset_class} ETF를 모아줘"),
]

def build_attribute_challenge(split, limit):
    """특정 ETF 이름 없이 운용사, 자산군, 시장 등 속성 조건만 제시된 Multi-Positive 검색 셋 구축"""
    seed_frame = etf_df[etf_df["split"] == split].sample(frac=1, random_state=SEED)
    candidates, seen = [], set()
    for fields, template in ATTRIBUTE_CHALLENGE_SPECS:
        for _, row in seed_frame.iterrows():
            values = tuple(row[field] for field in fields)
            if any(not str(value).strip() or str(value).strip() == "미상" for value in values):
                continue
            signature = (fields, values)
            if signature in seen:
                continue
            mask = pd.Series(True, index=etf_df.index)
            for field, value in zip(fields, values):
                mask &= etf_df[field] == value
            relevant = set(etf_df.loc[mask, "code"])
            if not 2 <= len(relevant) <= 60:
                continue
            seen.add(signature)
            payload = {field: value for field, value in zip(fields, values)}
            query_text = template.format(**payload)
            query_id = f"attribute:{split}:{len(candidates):03d}"
            candidates.append((query_id, query_text, relevant))

    random.Random(SEED + {"validation": 1, "test": 2}[split]).shuffle(candidates)
    selected = candidates[:limit]
    minimum = min(limit, 8)
    if len(selected) < minimum:
        raise RuntimeError(f"{split} attribute challenge 부족: {len(selected)} < {minimum}")
    return (
        {query_id: text for query_id, text, _ in selected},
        {query_id: relevant for query_id, _, relevant in selected},
    )

# validation 및 test 세트에 대한 Attribute Challenge 검색 셋 생성
attribute_queries, attribute_qrels = {}, {}
for split in ["validation", "test"]:
    attribute_queries[split], attribute_qrels[split] = build_attribute_challenge(split, CFG["ir_challenge_n"])
    relevant_counts = [len(rows) for rows in attribute_qrels[split].values()]
    print(f"attribute {split}: queries={len(attribute_queries[split])} | mean_relevant={np.mean(relevant_counts):.1f}")


**실행 결과 설명**

corpus 300개에서 entity query는 train 420, validation 90, test 90개가 만들어졌습니다. QUICK의 `ir_query_variants=2`이므로 ETF당 최대 두 종류의 entity 질문을 사용합니다.

attribute challenge는 validation/test 각각 24개이고 query당 관련 ETF가 평균 14.5/14.0개입니다. 다중 정답 검색이므로 Recall과 nDCG가 특히 중요합니다.


### **3-3. Hard negative·MNR·Triplet 데이터**

무작위 오답은 정답과 너무 달라 쉽게 구분됩니다. **Hard negative**는 baseline 검색기가 높은 점수를 줬지만 실제로는 오답인 문서라서 더 유익한 학습 신호가 됩니다.

- **MNR loss**: 한 batch의 다른 positive 문서를 서로의 negative로 사용해 정답 쌍의 상대 점수를 높임
- **Triplet**: `(query, positive, hard negative)`에서 positive가 negative보다 가깝도록 학습
- QUICK은 query마다 hard negative 3개를 선택

출력의 `MNR=420`, `triplet=1,260`은 query 420개 × hard negative 3개 관계입니다. negative가 실제 positive 집합과 겹치지 않는지 assertion으로 확인합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Baseline BGE-M3 임베딩 모델 기반 train 파티션 문서 인코딩
# -----------------------------------------------------------------------------
train_doc_ids = etf_df.loc[etf_df["split"] == "train", "code"].tolist()
train_doc_texts = [corpus[doc_id] for doc_id in train_doc_ids]
train_corpus_matrix = emb_model.encode(
    train_doc_texts,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

# -----------------------------------------------------------------------------
# 2. Hard Negative Mining (상위 코사인 유사도를 가진 오답 문서 마이닝)
# -----------------------------------------------------------------------------
mnr_rows, triplet_rows = [], []
train_hard_negatives = {}
for query_id, query_text in ir_queries["train"].items():
    # 질의 벡터 생성 및 train 내 전체 문서 벡터와의 내적 유사도 계산
    query_vector = emb_model.encode(query_text, convert_to_numpy=True, normalize_embeddings=True)
    scores = train_corpus_matrix @ query_vector
    ranking = np.argsort(-scores)
    positives = ir_qrels["train"][query_id]
    
    # 정답(Positive)이 아닌 문서 중 모델이 상위 유사도로 오인한 3개 문서 추출 (Hard Negatives)
    negatives = [train_doc_ids[index] for index in ranking if train_doc_ids[index] not in positives]
    negatives = negatives[:CFG["ir_hard_negatives"]]
    train_hard_negatives[query_id] = negatives
    
    positive_id = next(iter(positives))
    # MNR (Multiple Negatives Ranking) 용 (Anchor, Positive) 쌍
    mnr_rows.append({"anchor": query_text, "positive": corpus[positive_id]})
    
    # Triplet Loss 용 (Anchor, Positive, Negative) 삼중주 쌍
    for negative_id in negatives:
        triplet_rows.append({
            "anchor": query_text,
            "positive": corpus[positive_id],
            "negative": corpus[negative_id],
        })

# -----------------------------------------------------------------------------
# 3. HuggingFace Dataset 변환 및 Negative 셋 교차 누출 오염 검증
# -----------------------------------------------------------------------------
train_mnr = Dataset.from_list(mnr_rows)
train_triplet = Dataset.from_list(triplet_rows)
# 마이닝된 Hard Negative가 실제 Positive 정답 문서와 겹치지 않는지 교차 검증
assert all(set(train_hard_negatives[qid]).isdisjoint(ir_qrels["train"][qid]) for qid in train_hard_negatives)
print(f"MNR={len(train_mnr):,} | triplet={len(train_triplet):,}")


**실행 결과 설명**

MNR positive pair 420개와 triplet 1,260개가 생성됐습니다. 출력의 `Batches 0%`는 encoding progress의 초기 표시가 남은 것이며, 뒤의 데이터 개수 출력과 다음 평가가 진행됐으므로 mining은 완료됐습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 생성된 MNR 및 Triplet 데이터셋 샘플 구조 및 Hard Negative 비교 출력
# -----------------------------------------------------------------------------
print("=== [1. MNR Pair 데이터셋 샘플 (Anchor - Positive)] ===")
mnr_sample = train_mnr[0]
print(f"- Anchor (Query)  : {mnr_sample['anchor']}")
print(f"- Positive (Doc)  : {mnr_sample['positive'][:100]}...")

print("\n=== [2. Triplet 데이터셋 샘플 (Anchor - Positive - Hard Negative)] ===")
triplet_sample = train_triplet[0]
print(f"- Anchor (Query)   : {triplet_sample['anchor']}")
print(f"- Positive (Doc)   : {triplet_sample['positive'][:100]}...")
print(f"- Hard Negative    : {triplet_sample['negative'][:100]}...")


### **3-4. Ranking 지표를 작은 예제로 계산하기**

정답 문서가 `[A, C]`, 검색 순위가 `[B, A, D, C]`라고 가정합니다.

- **MRR**: 첫 정답 A가 2위이므로 $1/2=0.5$
- **Recall@4**: 정답 2개를 모두 찾았으므로 $2/2=1.0$
- **DCG**: 높은 순위의 정답에 더 큰 가중치를 주며 $\sum rel_i/\log_2(i+1)$로 계산
- **nDCG**: 실제 DCG를 이상적인 DCG로 나눠 0~1로 정규화
- **MAP**: 각 정답 위치에서의 precision을 평균

이 셀의 primary 평가는 전체 corpus 순위인 MRR/nDCG/Recall입니다. Binary AP와 Spearman은 선택한 positive/negative 쌍의 분리 정도를 보는 **보조 진단**입니다. pair 지표가 좋아져도 전체 검색 순위가 나빠질 수 있습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. InformationRetrievalEvaluator 객체 생성 함수 (1024, 512, 128 차원 축소 지원)
# -----------------------------------------------------------------------------
def make_ir_evaluator(split, dimension, challenge=False):
    """SentenceTransformers IR 평가기 생성 (MRR@10, nDCG@10, Recall@10 측정)"""
    queries = attribute_queries[split] if challenge else ir_queries[split]
    qrels = attribute_qrels[split] if challenge else ir_qrels[split]
    family = "attribute" if challenge else "entity"
    return InformationRetrievalEvaluator(
        queries=queries,
        corpus=corpus,
        relevant_docs=qrels,
        mrr_at_k=[10],
        ndcg_at_k=[10],
        precision_recall_at_k=[10],
        map_at_k=[10],
        name=f"etf_{family}_{split}_dim{dimension}",
        truncate_dim=dimension,  # Matryoshka 평가용 벡터 차원 슬라이싱
        show_progress_bar=True,
        write_csv=False,
    )

# 차원별(1024, 512, 128) test 셋 및 challenge 셋 평가기 딕셔너리 구성
ir_test_evaluators = {dimension: make_ir_evaluator("test", dimension) for dimension in EMBEDDING_DIMS}
ir_challenge_test_evaluators = {dimension: make_ir_evaluator("test", dimension, challenge=True) for dimension in EMBEDDING_DIMS}
ir_validation_evaluator = make_ir_evaluator("validation", 1024, challenge=True)

# -----------------------------------------------------------------------------
# 2. 보조 진단 지표용 Binary/Spearman 쌍(Pair) 데이터 구축
# -----------------------------------------------------------------------------
# test query마다 positive 1개 + baseline top hard negative 1개 추출
test_pair_rows = []
corpus_ids = list(corpus)
corpus_matrix = emb_model.encode(
    [corpus[doc_id] for doc_id in corpus_ids],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)
for query_id, query_text in ir_queries["test"].items():
    positive_id = next(iter(ir_qrels["test"][query_id]))
    vector = emb_model.encode(query_text, convert_to_numpy=True, normalize_embeddings=True)
    ranking = np.argsort(-(corpus_matrix @ vector))
    negative_id = next(corpus_ids[index] for index in ranking if corpus_ids[index] != positive_id)
    test_pair_rows.extend([
        {"sentence1": query_text, "sentence2": corpus[positive_id], "label": 1},
        {"sentence1": query_text, "sentence2": corpus[negative_id], "label": 0},
    ])

binary_evaluator = BinaryClassificationEvaluator(
    sentences1=[row["sentence1"] for row in test_pair_rows],
    sentences2=[row["sentence2"] for row in test_pair_rows],
    labels=[row["label"] for row in test_pair_rows],
    name="etf_pair_ap",
    write_csv=False,
)
spearman_evaluator = EmbeddingSimilarityEvaluator(
    sentences1=[row["sentence1"] for row in test_pair_rows],
    sentences2=[row["sentence2"] for row in test_pair_rows],
    scores=[float(row["label"]) for row in test_pair_rows],
    name="etf_pair_spearman",
    write_csv=False,
)

# -----------------------------------------------------------------------------
# 3. 평가 결과 파싱 헬퍼 및 Baseline(학습 전) 평가 실행 및 표 출력
# -----------------------------------------------------------------------------
def find_metric(result, token):
    """결과 딕셔너리에서 대소문자 구분 없이 특정 지표 키 값 추출"""
    matches = [float(value) for key, value in result.items() if token.lower() in key.lower() and isinstance(value, numbers.Real)]
    return matches[0] if matches else float("nan")

def ir_metric_rows(stage, family, results):
    """MRR@10, nDCG@10, Recall@10 지표를 테이블 구조체로 정리"""
    rows = []
    for dimension, result in results.items():
        for metric in ["mrr@10", "ndcg@10", "recall@10"]:
            rows.append({"stage": stage, "family": family, "dimension": dimension, "metric": metric, "value": find_metric(result, metric)})
    return rows

# 파인튜닝 학습 전(Baseline) 임베딩 성능 측정
embedding_before_ir = {dimension: evaluator(emb_model) for dimension, evaluator in ir_test_evaluators.items()}
embedding_before_challenge_ir = {dimension: evaluator(emb_model) for dimension, evaluator in ir_challenge_test_evaluators.items()}
embedding_before_validation = ir_validation_evaluator(emb_model)
embedding_before_secondary = {
    "binary": binary_evaluator(emb_model),
    "spearman": spearman_evaluator(emb_model),
}
baseline_ir_table = pd.DataFrame(
    ir_metric_rows("Before", "entity", embedding_before_ir)
    + ir_metric_rows("Before", "attribute_challenge", embedding_before_challenge_ir)
)
print("\nEmbedding baseline ranking metrics")
print(baseline_ir_table.round(4).to_string(index=False))


**Baseline 결과 설명**

Entity 검색은 1024·512·128차원에서 MRR/nDCG가 약 0.99~1.0이고 Recall@10은 모두 1.0입니다. 상품명이 포함된 질문은 pretrained BGE-M3만으로도 거의 해결됐습니다.

반면 1024차원 attribute challenge는 MRR 0.6036, nDCG 0.4311, Recall 0.3564입니다. 차원이 512·128로 줄수록 더 낮아져, 이름 없는 속성 검색이 실제 개선 여지가 있는 핵심 평가임을 보여줍니다.


In [ ]:
# -----------------------------------------------------------------------------
# Baseline BGE-M3의 Attribute Challenge (속성 검색) 예시 케이스 및 탑 레트리벌 출력
# -----------------------------------------------------------------------------
# 1. Attribute Challenge Test 세트의 첫 번째 예시 질의 선택
sample_qid = list(attribute_queries["test"])[0]
sample_query = attribute_queries["test"][sample_qid]
sample_qrels = attribute_qrels["test"][sample_qid]

print(f"=== [Query ID: {sample_qid}] ===")
print(f"- 검색 질의 (Attribute Query): {sample_query}")
print(f"- 정답 ETF 종목 수 (Relevant Docs): {len(sample_qrels)}개 (예: {list(sample_qrels)[:3]})")

# 2. Baseline 모델로 검색 질의 및 전체 Corpus 인코딩 후 코사인 유사도 탑 5 검색
q_vec = emb_model.encode(sample_query, convert_to_numpy=True, normalize_embeddings=True)
doc_ids = list(corpus)
doc_matrix = emb_model.encode([corpus[did] for did in doc_ids], convert_to_numpy=True, normalize_embeddings=True)
top_indices = np.argsort(-(doc_matrix @ q_vec))[:5]

print("\n=== [Baseline BGE-M3 Top-5 검색 결과] ===")
for rank, idx in enumerate(top_indices, 1):
    did = doc_ids[idx]
    is_hit = "✅ [HIT]" if did in sample_qrels else "❌ [MISS]"
    print(f"{rank}위 {is_hit} (코드: {did}) -> {corpus[did][:90]}...")


### **3-5. Matryoshka Embedding 학습**

Matryoshka는 큰 인형 안에 작은 인형이 들어가듯, 앞쪽 128·512차원만 잘라 사용해도 의미 구조가 남도록 여러 차원에서 동시에 loss를 계산합니다.

- MNR 데이터와 triplet 데이터를 round-robin으로 번갈아 학습
- 중복 없는 batch sampler로 같은 문장이 가짜 negative가 되는 문제 완화
- device batch 32, accumulation 1 → **유효 배치 32**
- validation attribute nDCG@10이 가장 좋은 checkpoint 선택

Embedding 모델은 LoRA가 아니라 전체 567.8M 파라미터를 학습하므로 Part 1·2의 trainable 비율과 직접 비교하면 안 됩니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. MultipleNegativesRankingLoss 및 Matryoshka (1024, 512, 128) 손실 함수 조합
# -----------------------------------------------------------------------------
base_loss = MultipleNegativesRankingLoss(emb_model)
matryoshka_loss = MatryoshkaLoss(
    emb_model,
    base_loss,
    matryoshka_dims=EMBEDDING_DIMS,  # [1024, 512, 128] 차원별 손실 합산
)

# -----------------------------------------------------------------------------
# 2. SentenceTransformerTrainer 파라미터 및 하이브리드 데이터셋 설정
# -----------------------------------------------------------------------------
emb_trainer = SentenceTransformerTrainer(
    model=emb_model,
    train_dataset={"mnr": train_mnr, "triplet": train_triplet},  # MNR + Triplet 하이브리드 학습
    loss={"mnr": matryoshka_loss, "triplet": matryoshka_loss},
    evaluator=ir_validation_evaluator,                           # Validation Attribute Challenge 평가기
    args=SentenceTransformerTrainingArguments(
        output_dir=str(WORK_DIR / "bge_m3_checkpoints"),        # 체크포인트 저장 경로
        num_train_epochs=CFG["embedding_epochs"],               # Epoch 수 (2)
        per_device_train_batch_size=32,                         # 디바이스 당 배치 크기 (32)
        learning_rate=2e-5,                                     # 학습률 (2e-5)
        warmup_ratio=0.10,                                      # 웜업 비율 (10%)
        bf16=True,                                              # BF16 연산 가속
        batch_sampler=BatchSamplers.NO_DUPLICATES,              # 배치 내 가짜 Negative 방지 샘플러
        multi_dataset_batch_sampler=MultiDatasetBatchSamplers.ROUND_ROBIN, # MNR과 Triplet 교차 라운드로빈
        eval_strategy="epoch",                                  # Epoch 마다 검증 수행
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,                            # 최고 성능 검증 모델 복원
        metric_for_best_model="eval_etf_attribute_validation_dim1024_cosine_ndcg@10", # 타겟 지표 (nDCG@10)
        greater_is_better=True,                                 # nDCG@10은 클수록 우수
        logging_steps=10,
        report_to="none",
        seed=SEED,
    ),
)

# -----------------------------------------------------------------------------
# 3. Part 3 BGE-M3 임베딩 파인튜닝 학습 진행
# -----------------------------------------------------------------------------
print("🚀 Part 3 Embedding FT 시작")
emb_stats = emb_trainer.train()
print("✅ train_loss=", emb_stats.training_loss)


In [ ]:
# -----------------------------------------------------------------------------
# Part 3 임베딩 모델 학습 진행에 따른 Loss 커브 및 Validation nDCG@10 시각화
# -----------------------------------------------------------------------------
import matplotlib.pyplot as plt

# 1. Trainer의 log_history에서 Step별 Training Loss 및 Validation nDCG 기록 추출
history = emb_trainer.state.log_history
loss_steps = [log["step"] for log in history if "loss" in log]
losses = [log["loss"] for log in history if "loss" in log]

val_epochs = [log["epoch"] for log in history if "eval_etf_attribute_validation_dim1024_cosine_ndcg@10" in log]
val_ndcg = [log["eval_etf_attribute_validation_dim1024_cosine_ndcg@10"] for log in history if "eval_etf_attribute_validation_dim1024_cosine_ndcg@10" in log]

# 2. Subplot 그래프 시각화 (좌: Training Loss 커브 / 우: Validation nDCG@10 평가 지표)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# (1) Training Loss 커브 플롯
if loss_steps:
    ax1.plot(loss_steps, losses, marker="o", color="#1f77b4", linewidth=2, label="Training Loss")
    ax1.set_title("BGE-M3 Training Loss Curve", fontsize=12, fontweight="bold")
    ax1.set_xlabel("Training Steps")
    ax1.set_ylabel("Loss")
    ax1.grid(True, linestyle="--", alpha=0.6)
    ax1.legend()
else:
    ax1.text(0.5, 0.5, "Loss history not found", ha="center", va="center")

# (2) Validation nDCG@10 커브 플롯
if val_epochs:
    ax2.plot(val_epochs, val_ndcg, marker="s", color="#2ca02c", linewidth=2, label="Val nDCG@10 (1024-dim)")
    ax2.set_title("Validation Attribute nDCG@10", fontsize=12, fontweight="bold")
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("nDCG@10 Score")
    ax2.grid(True, linestyle="--", alpha=0.6)
    ax2.legend()
else:
    ax2.text(0.5, 0.5, "Validation history not found", ha="center", va="center")

plt.tight_layout()
plt.show()


**실행 결과 설명**

- 학습 예제 1,680 = MNR 420 + triplet 1,260
- 2 epoch, 56 optimizer step, 유효 배치 32
- 전체 567,754,752 파라미터 학습
- 최종 train loss 약 0.4553

loss가 정상적으로 계산되고 학습이 완료됐지만, embedding 품질은 loss 값이 아니라 다음 셀의 전체 corpus ranking으로 판정해야 합니다.


### **3-6. 쉬운 Entity와 어려운 Attribute 결과 분리**

Before/After 표는 `family × dimension × metric` 조합마다 변화량을 보여줍니다.

- Entity가 이미 1.0에 가깝다면 개선 여지가 거의 없는 **ceiling effect**가 생깁니다.
- Attribute는 상품명 없이 속성으로 여러 정답을 찾아야 하므로 domain 일반화를 더 잘 드러냅니다.
- 작은 차원까지 모두 하락하면 단순 차원 축소 문제가 아니라 학습 목표·negative 구성·데이터 누수를 의심해야 합니다.

Primary ranking과 secondary pair 지표가 반대 방향이면 실제 서비스 동작과 가까운 전체 corpus ranking을 우선합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. 파인튜닝 후(After FT) BGE-M3 임베딩 모델의 IR 지표 및 Secondary 평가
# -----------------------------------------------------------------------------
embedding_after_ir = {dimension: evaluator(emb_model) for dimension, evaluator in ir_test_evaluators.items()}
embedding_after_challenge_ir = {dimension: evaluator(emb_model) for dimension, evaluator in ir_challenge_test_evaluators.items()}
embedding_after_secondary = {
    "binary": binary_evaluator(emb_model),
    "spearman": spearman_evaluator(emb_model),
}

# -----------------------------------------------------------------------------
# 2. 파인튜닝 전/후 (Before vs After) 차원별 검색 성능 증감(delta) 정리
# -----------------------------------------------------------------------------
rows = []
for family, before_results, after_results in [
    ("entity", embedding_before_ir, embedding_after_ir),
    ("attribute_challenge", embedding_before_challenge_ir, embedding_after_challenge_ir),
]:
    for dimension in EMBEDDING_DIMS:
        for metric in ["mrr@10", "ndcg@10", "recall@10"]:
            before_value = find_metric(before_results[dimension], metric)
            after_value = find_metric(after_results[dimension], metric)
            rows.append({
                "family": family, "dimension": dimension, "metric": metric,
                "before": before_value, "after": after_value, "delta": after_value - before_value,
            })
embedding_comparison = pd.DataFrame(rows)
print(embedding_comparison.round(4).to_string(index=False))
print("\nSecondary Before:", embedding_before_secondary)
print("Secondary After :", embedding_after_secondary)

# -----------------------------------------------------------------------------
# 3. 파인튜닝 지표 저장 및 학습된 Matryoshka 임베딩 모델 로컬/허브 저장
# -----------------------------------------------------------------------------
embedding_metrics_path = WORK_DIR / "embedding_eval_metrics.json"
with embedding_metrics_path.open("w", encoding="utf-8") as file:
    json.dump({
        "ranking": rows,
        "secondary": {"before": embedding_before_secondary, "after": embedding_after_secondary},
    }, file, ensure_ascii=False, indent=2, default=float)

embedding_save_path = WORK_DIR / "bge_m3_etf_finetuned"
emb_model.save(str(embedding_save_path))  # SentenceTransformer 파인튜닝 임베딩 저장
if PUSH_TO_HUB:
    emb_model.push_to_hub(
        f"{HF_NAMESPACE}/bge-m3-etf-matryoshka",
        token=HF_TOKEN,
        private=HF_PRIVATE,
        exist_ok=True,
    )
print(f"✅ Embedding={embedding_save_path} | metrics={embedding_metrics_path}")


**실행 결과 설명 — 왜 Part 3은 `FAIL`인가?**

1024차원 attribute challenge의 핵심 변화는 다음과 같습니다.

| 지표 | Before | After | 변화 |
|---|---:|---:|---:|
| MRR@10 | 0.6036 | 0.3313 | -0.2722 |
| nDCG@10 | 0.4311 | 0.2352 | -0.1959 |
| Recall@10 | 0.3564 | 0.1467 | -0.2097 |

반대로 pair AP는 `0.8480→0.9996`, Spearman은 `0.5778→0.8654`로 올랐습니다. 선택된 한 positive/negative 쌍은 잘 분리하지만 전체 300개 문서를 올바르게 정렬하지 못한다는 뜻입니다.

**결론**: 이 FT embedding은 검색 승격 대상이 아니며 Part 4의 Stage 1에는 pretrained BGE-M3 fallback이 더 안전합니다.


### **3-7. 정성 검색 예시를 볼 때의 함정**

한 query의 Top-10을 직접 보는 것은 문서 내용과 오류 유형을 이해하는 데 유용합니다. 하지만 이 데모는 상품명이 포함된 쉬운 entity query이며 After 결과만 출력합니다.

따라서 “정답이 1위다”라는 한 사례로 전체 성능 향상을 결론내리면 안 됩니다. 앞의 attribute challenge 집계 결과와 실패 query의 Before/After 순위를 함께 봐야 합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 정답 문서(Gold Positive)가 존재하는 test query의 Top-10 검색 결과 정성 확인
# -----------------------------------------------------------------------------
# 1. Test 파티션에서 첫 번째 테스트 질의 및 정답 종목 코드 추출
demo_query_id = next(iter(ir_queries["test"]))
demo_query = ir_queries["test"][demo_query_id]
demo_positive = next(iter(ir_qrels["test"][demo_query_id]))

# 2. 파인튜닝된 BGE-M3 모델로 질의 및 전체 Corpus 인코딩 후 코사인 유사도 연산
demo_vector = emb_model.encode(demo_query, convert_to_numpy=True, normalize_embeddings=True)
demo_matrix = emb_model.encode(list(corpus.values()), convert_to_numpy=True, normalize_embeddings=True)
demo_ids = list(corpus)
demo_ranking = np.argsort(-(demo_matrix @ demo_vector))[:10]

# 3. 검색 1위~10위 문서 출력 및 정답 문서(Gold Positive) 체크리스트 표시 (✅)
print("Query:", demo_query)
print("Gold :", demo_positive)
for rank, index in enumerate(demo_ranking, 1):
    marker = "✅" if demo_ids[index] == demo_positive else "  "
    print(f"{marker} {rank:>2}위 {demo_ids[index]} | {corpus[demo_ids[index]][:110]}")


**실행 결과 설명**

예시 query의 gold `241390`이 1위에 있어 눈으로 보기에는 성공입니다. 그러나 상품명이 그대로 포함된 쉬운 entity query라서 앞의 attribute 회귀를 드러내지 못합니다.

정성 예시는 “어떤 문서가 왜 비슷하게 검색됐는가”를 이해하는 보조 자료로 사용하고, 전체 모델 판정은 24개 attribute query의 집계 지표를 따릅니다.


### Part 3 실행 결과 해석 — `FAIL`, pretrained fallback 필요

- Entity 검색은 pretrained 단계부터 거의 포화됐고 FT 후 1024/512/128 차원에서 모두 1.0에 가까웠습니다.
- 반면 핵심 attribute challenge 1024차원은 MRR@10 0.6036→0.3313, nDCG@10 0.4311→0.2352, Recall@10 0.3564→0.1467로 크게 하락했습니다.
- Pair AP는 0.8480→0.9996, Spearman은 0.5778→0.8654로 올랐지만 pair 분리와 corpus ranking은 다른 문제임을 보여줍니다.
- 기존 attribute 생성은 validation/test 각각 local `seen`을 사용해 7/24 signature/text가 겹쳤으므로 평가 독립성도 보강해야 합니다.
- 마지막 demo는 이미 쉬운 exact-name query의 After 순위만 출력해 회귀를 드러내지 못했습니다.

이 FT embedding은 ranking promotion gate를 통과하지 못하며, Part 4의 Stage 1은 pretrained BGE-M3를 사용하는 것이 안전합니다.


---

# Part 4: Reranker FT — ETF 2단계 정밀 검색

| 구조 | 계산 | 장점 | 역할 |
|---|---|---|---|
| Bi-Encoder | query/doc를 독립 인코딩 | 전체 corpus 고속 검색 | Top-K 후보 생성 |
| Cross-Encoder | `[query; doc]`를 함께 인코딩 | 문맥 기반 정밀 점수 | 후보 재정렬 |

```text
Query → BGE-M3 전체 검색 → Top-K 후보 → BGE Reranker → Top-N
```

Reranker는 쉬운 random negative보다 실제 retriever가 높은 순위로 잘못 가져온 hard negative에서 더 많이 배웁니다. 같은 자산군·운용사의 near-negative를 우선하고, train/validation/test 후보군은 고정해 Before/After를 동일하게 평가합니다.

최종 평가는 exact-name entity query와 이름 없는 multi-positive attribute challenge를 모두 포함합니다. Stage 1 후보 누락은 attribute Recall@K로, 후보 내부 순위는 reranker nDCG로 나눠 측정하고 함께 해석합니다.


### **4-1. 고정 Stage 1 후보와 Reranker 학습쌍**

2-stage 검색은 역할을 나눕니다.

```text
Query → Bi-Encoder로 전체 300개 검색 → Top-20 후보 → Cross-Encoder 재정렬
```

Reranker는 Stage 1이 가져온 후보만 볼 수 있으므로 정답이 Top-K 밖이면 복구할 수 없습니다. 그래서 후보 목록을 Before/After 사이에 고정하고 Stage 1 Recall@K와 Stage 2 nDCG를 분리합니다.

학습 negative는 같은 자산군이나 같은 운용사처럼 헷갈리기 쉬운 near-negative를 먼저 사용합니다. positive 1개와 hard negative 3개이므로 query당 4개의 labeled pair가 만들어집니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Stage 1 Retriever (BGE-M3) 기반 고정 상위 Top-K 후보(Candidate) 추출
# -----------------------------------------------------------------------------
all_doc_ids = list(corpus)
all_doc_matrix = emb_model.encode(
    [corpus[doc_id] for doc_id in all_doc_ids],
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
)

def stage1_candidates(split, restrict_to_split):
    """지정된 split에서 Stage 1 1차 임베딩 모델로 상위 Top-K(20개) 후보 문서 랭킹 수집"""
    if restrict_to_split:
        allowed_ids = set(etf_df.loc[etf_df["split"] == split, "code"])
        candidate_ids = [doc_id for doc_id in all_doc_ids if doc_id in allowed_ids]
        indices = [all_doc_ids.index(doc_id) for doc_id in candidate_ids]
        matrix = all_doc_matrix[indices]
    else:
        candidate_ids = all_doc_ids
        matrix = all_doc_matrix
    result = {}
    for query_id, query_text in ir_queries[split].items():
        vector = emb_model.encode(query_text, convert_to_numpy=True, normalize_embeddings=True)
        ranking = np.argsort(-(matrix @ vector))[:CFG["candidate_k"]]
        result[query_id] = [candidate_ids[index] for index in ranking]
    return result

fixed_candidates = {
    "train": stage1_candidates("train", restrict_to_split=True),
    "validation": stage1_candidates("validation", restrict_to_split=True),
    "test": stage1_candidates("test", restrict_to_split=False),
}

def stage1_attribute_candidates(split):
    """Attribute Challenge 셋에 대한 1차 레트리벌 상위 Top-K 후보 추출"""
    result = {}
    for query_id, query_text in attribute_queries[split].items():
        vector = emb_model.encode(query_text, convert_to_numpy=True, normalize_embeddings=True)
        ranking = np.argsort(-(all_doc_matrix @ vector))[:CFG["candidate_k"]]
        result[query_id] = [all_doc_ids[index] for index in ranking]
    return result

attribute_fixed_candidates = {
    split: stage1_attribute_candidates(split) for split in ["validation", "test"]
}
etf_lookup = etf_df.set_index("code")

# -----------------------------------------------------------------------------
# 2. Reranker 파인튜닝용 Near-Hard Negative 정렬 및 Labeled Pair 데이터 생성
# -----------------------------------------------------------------------------
def hard_negative_order(positive_id, ranked_ids):
    """동일 자산군/운용사 등 오해하기 쉬운 Near-Negative를 우선 정렬"""
    positive = etf_lookup.loc[positive_id]
    near_ids = set(etf_df.loc[
        ((etf_df["asset_class"] == positive["asset_class"]) | (etf_df["manager"] == positive["manager"]))
        & (etf_df["code"] != positive_id),
        "code",
    ])
    return [doc_id for doc_id in ranked_ids if doc_id in near_ids] + [doc_id for doc_id in ranked_ids if doc_id not in near_ids]

reranker_train_rows = []
for query_id, query_text in ir_queries["train"].items():
    positive_ids = ir_qrels["train"][query_id]
    positive_id = next(iter(positive_ids))
    ranked_negatives = [doc_id for doc_id in fixed_candidates["train"][query_id] if doc_id not in positive_ids]
    negatives = hard_negative_order(positive_id, ranked_negatives)
    negatives = negatives[:CFG["ir_hard_negatives"]]
    reranker_train_rows.append({"query": query_text, "document": corpus[positive_id], "label": 1.0})
    reranker_train_rows.extend({"query": query_text, "document": corpus[doc_id], "label": 0.0} for doc_id in negatives)

reranker_train_ds = Dataset.from_list(reranker_train_rows)
assert reranker_train_ds.column_names == ["query", "document", "label"]
assert set(reranker_train_ds["label"]) == {0.0, 1.0}

# -----------------------------------------------------------------------------
# 3. Validation 및 Test 세트용 Reranking 평가 샘플 생성기
# -----------------------------------------------------------------------------
def reranking_samples(split, realistic):
    """Cross-Encoder Reranking Evaluator 평가용 질의-문서 묶음 샘플 구성"""
    samples = []
    for query_id, query_text in ir_queries[split].items():
        positive_ids = ir_qrels[split][query_id]
        positives = [corpus[doc_id] for doc_id in positive_ids]
        ranked_ids = fixed_candidates[split][query_id]
        if realistic:
            samples.append({
                "query": query_text,
                "positive": positives,
                "documents": [corpus[doc_id] for doc_id in ranked_ids],
            })
        else:
            negatives = [corpus[doc_id] for doc_id in ranked_ids if doc_id not in positive_ids]
            samples.append({"query": query_text, "positive": positives, "negative": negatives})
    return samples

reranker_validation_samples = reranking_samples("validation", realistic=False)
reranker_test_samples = reranking_samples("test", realistic=True)

def attribute_reranking_samples(split):
    """Attribute Challenge 셋용 Reranking 샘플 생성"""
    samples = []
    for query_id, query_text in attribute_queries[split].items():
        positive_ids = attribute_qrels[split][query_id]
        samples.append({
            "query": query_text,
            "positive": [corpus[doc_id] for doc_id in positive_ids],
            "documents": [corpus[doc_id] for doc_id in attribute_fixed_candidates[split][query_id]],
        })
    return samples

reranker_attribute_validation_samples = attribute_reranking_samples("validation")
reranker_attribute_test_samples = attribute_reranking_samples("test")
print(
    f"reranker train={len(reranker_train_ds):,} | entity_test={len(reranker_test_samples)} | "
    f"attribute_val/test={len(reranker_attribute_validation_samples)}/{len(reranker_attribute_test_samples)}"
)


**실행 결과 설명**

reranker 학습쌍은 1,680개이며 entity test query 90개, attribute validation/test 24개씩의 고정 후보가 준비됐습니다. `train=1,680`은 query 420개 × (positive 1 + negative 3)입니다.

후보를 고정했으므로 이후 점수 변화는 Stage 1 후보가 달라진 효과가 아니라 Reranker 순서 변화로 해석할 수 있습니다.


In [ ]:
# -----------------------------------------------------------------------------
# 생성된 Reranker 훈련용 Dataset 샘플 (Positive vs Near-Hard Negative) 출력
# -----------------------------------------------------------------------------
print("=== [1. Reranker Positive (Label 1.0) 데이터 샘플] ===")
pos_sample = next(row for row in reranker_train_ds if row["label"] == 1.0)
print(f"- Query   : {pos_sample['query']}")
print(f"- Document: {pos_sample['document'][:100]}...")
print(f"- Label   : {pos_sample['label']}")

print("\n=== [2. Reranker Hard Negative (Label 0.0) 데이터 샘플] ===")
neg_sample = next(row for row in reranker_train_ds if row["label"] == 0.0)
print(f"- Query   : {neg_sample['query']}")
print(f"- Document: {neg_sample['document'][:100]}...")
print(f"- Label   : {neg_sample['label']}")


### **4-2. Cross-Encoder와 off-the-shelf baseline**

| 모델 | 입력 방식 | 장점 | 단점 |
|---|---|---|---|
| Bi-Encoder | query와 문서를 따로 인코딩 | 문서 벡터 재사용, 빠름 | 두 텍스트의 세밀한 상호작용 제한 |
| Cross-Encoder | `[query; document]`를 함께 입력 | 관련성을 정밀하게 판단 | 후보마다 모델 실행, 느림 |

먼저 학습하지 않은 `bge-reranker-v2-m3`를 고정 후보에 적용해 baseline을 측정합니다. 그래야 “reranker 자체의 효과”와 “ETF 데이터로 추가 FT한 효과”를 분리할 수 있습니다.

출력에 보이는 0% 다운로드 줄은 Jupyter가 진행 표시의 중간 상태를 저장한 흔적입니다. 뒤의 `baseline 완료`가 출력되고 다음 셀이 실행됐으므로 다운로드 실패가 아닙니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Part 3 임베딩 모델 객체 삭제 및 VRAM 해제
# -----------------------------------------------------------------------------
del emb_trainer, emb_model, matryoshka_loss, base_loss
gc.collect()
torch.cuda.empty_cache()

# -----------------------------------------------------------------------------
# 2. CrossEncoder Reranker 모델 (bge-reranker-v2-m3) 및 평가기 설정
# -----------------------------------------------------------------------------
from sentence_transformers.cross_encoder import (
    CrossEncoder,
    CrossEncoderTrainer,
    CrossEncoderTrainingArguments,
)
from sentence_transformers.cross_encoder.evaluation import CrossEncoderRerankingEvaluator
from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

# BAAI/bge-reranker-v2-m3 Cross-Encoder 사전학습 모델 로드
RERANKER_MODEL = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(
    RERANKER_MODEL,
    num_labels=1,                           # 단일 유사도 연관성 점수(Logit) 출력
    max_length=CFG["reranker_max_length"],   # 최대 512 시퀀스 길이
    model_kwargs={"dtype": torch.float32},  # Float32 수치 정밀도
)

# -----------------------------------------------------------------------------
# 3. Validation / Test 셋에 대한 CrossEncoderRerankingEvaluator 생성
# -----------------------------------------------------------------------------
reranker_validation_evaluator = CrossEncoderRerankingEvaluator(
    samples=reranker_attribute_validation_samples,
    at_k=10,
    always_rerank_positives=False,
    name="etf_attribute_reranker_validation",
    batch_size=16,
    show_progress_bar=True,
    write_csv=False,
)
reranker_test_evaluator = CrossEncoderRerankingEvaluator(
    samples=reranker_test_samples,
    at_k=10,
    always_rerank_positives=False,
    name="etf_two_stage_test",
    batch_size=16,
    show_progress_bar=True,
    write_csv=False,
)
reranker_attribute_test_evaluator = CrossEncoderRerankingEvaluator(
    samples=reranker_attribute_test_samples,
    at_k=10,
    always_rerank_positives=False,
    name="etf_attribute_two_stage_test",
    batch_size=16,
    show_progress_bar=True,
    write_csv=False,
)

# -----------------------------------------------------------------------------
# 4. Off-the-shelf 사전학습(Baseline) Reranker 평가 실행
# -----------------------------------------------------------------------------
reranker_before_validation = reranker_validation_evaluator(reranker)
reranker_before_test = reranker_test_evaluator(reranker)
reranker_before_attribute_test = reranker_attribute_test_evaluator(reranker)
print("✅ Off-the-shelf reranker baseline 완료")


**실행 결과 설명**

약 2.27GB 모델과 tokenizer 파일을 내려받은 뒤 off-the-shelf baseline 평가가 완료됐습니다. 

이 baseline이 이미 attribute 순위를 크게 개선하는지 확인해야 추가 FT의 진짜 기여분을 계산할 수 있습니다.


In [ ]:
# -----------------------------------------------------------------------------
# Off-the-shelf 사전학습(Baseline) Reranker의 평가 지표 (MRR@10, nDCG@10, MAP@10) 출력
# -----------------------------------------------------------------------------
def parse_reranker_results(name, result_dict):
    """CrossEncoderRerankingEvaluator 결과 딕셔너리에서 주요 랭킹 지표 파싱"""
    rows = []
    for key, val in result_dict.items():
        if isinstance(val, (int, float)):
            rows.append({"Evaluator": name, "Metric": key, "Score": round(val, 4)})
    return pd.DataFrame(rows)

base_val_df = parse_reranker_results("Baseline Validation Attribute", {"score": reranker_before_validation})
base_test_df = parse_reranker_results("Baseline Test Entity", {"score": reranker_before_test})
base_attr_test_df = parse_reranker_results("Baseline Test Attribute", {"score": reranker_before_attribute_test})

print("=== [Off-the-shelf Baseline Reranker 평가 결과] ===")
print(f"- Validation Attribute Score: {reranker_before_validation:.4f}")
print(f"- Test Entity Score        : {reranker_before_test:.4f}")
print(f"- Test Attribute Score     : {reranker_before_attribute_test:.4f}")


### **4-3. Weighted BCE로 Reranker 학습하기**

Cross-Encoder는 query-document 쌍에 관련 점수 하나를 출력하고 Binary Cross Entropy로 positive(1)와 negative(0)를 구분합니다.

$$L=-w_+y\log\sigma(z)-(1-y)\log(1-\sigma(z))$$

query마다 negative가 3개라 positive가 상대적으로 적습니다. `pos_weight=3`은 positive 오류의 손실을 세 배로 주어 class imbalance를 보완합니다.

- device batch 8 × accumulation 2 = **유효 배치 16**
- learning rate $1\times10^{-5}$
- validation attribute nDCG@10으로 checkpoint 선택
- 이 모델은 전체 약 567.8M 파라미터를 학습


In [ ]:
# -----------------------------------------------------------------------------
# 1. Class Imbalance 보완용 Weighted BCE Loss 및 Reranker TrainingArguments 설정
# -----------------------------------------------------------------------------
n_negatives = CFG["ir_hard_negatives"]  # Query 당 Negative 개수 (3개)

# Positive 1개 : Negative 3개 불균형 보완을 위한 pos_weight=3.0 가중치 BCE 손실 함수
reranker_loss = BinaryCrossEntropyLoss(
    reranker,
    pos_weight=torch.tensor(float(n_negatives)),
)

# CrossEncoderTrainer 하이퍼파라미터 설정 (유효 배치 = 8x2 = 16)
reranker_args = CrossEncoderTrainingArguments(
    output_dir=str(WORK_DIR / "bge_reranker_checkpoints"),  # 체크포인트 저장 경로
    num_train_epochs=1,                                    # 1 Epoch 훈련
    per_device_train_batch_size=8,                         # 디바이스 당 배치 수 (8)
    per_device_eval_batch_size=16,                         # 평가 배치 수 (16)
    gradient_accumulation_steps=2,                         # 그래디언트 누적 (2)
    learning_rate=1e-5,                                    # 학습률 (1e-5)
    warmup_ratio=0.10,                                     # 웜업 비율 (10%)
    bf16=True,                                             # BF16 연산 가속
    gradient_checkpointing=True,                           # 메모리 절약 체크포인팅
    eval_strategy="epoch",                                 # Epoch 마다 검증 수행
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,                           # 최우수 검증 모델 복원
    metric_for_best_model="eval_etf_attribute_reranker_validation_ndcg@10", # 타겟 랭킹 지표 (nDCG@10)
    greater_is_better=True,
    logging_steps=10,
    report_to="none",
    seed=SEED,
)

# -----------------------------------------------------------------------------
# 2. CrossEncoderTrainer 객체 생성 및 Part 4 Reranker 파인튜닝 학습 실행
# -----------------------------------------------------------------------------
reranker_trainer = CrossEncoderTrainer(
    model=reranker,
    args=reranker_args,
    train_dataset=reranker_train_ds,
    loss=reranker_loss,
    evaluator=reranker_validation_evaluator,
)

print("🚀 Part 4 Reranker FT 시작")
reranker_stats = reranker_trainer.train()
print("✅ train_loss=", reranker_stats.training_loss)


**실행 결과 설명**

- 예제 1,680, 1 epoch, 105 step
- device batch 8 × accumulation 2 = 유효 배치 16
- 전체 567,755,777 파라미터 학습
- train loss 약 0.04012

낮은 train loss는 학습쌍 분리가 쉬워졌다는 신호일 수 있지만 과적합 여부와 전체 순위 개선은 다음 평가 지표로 확인해야 합니다.


### **4-4. Baseline과 FT Reranker 비교·저장·Smoke Test**

이 셀은 같은 고정 후보에서 off-the-shelf Before와 FT After를 비교합니다.

- `base_*`: Reranker를 적용하기 전 Stage 1 순서
- `map`, `mrr@10`, `ndcg@10`: Reranker가 후보 내부 순서를 얼마나 개선했는지
- `smoke`: 저장한 모델을 다시 불러 positive와 negative에 유한한 점수를 내는지 확인

smoke의 `0.99998`과 `0.00001`은 선택한 한 쌍의 분리 점수입니다. 전체 데이터에 대해 보정된 확률이나 품질 보증이 아닙니다.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Reranker 파인튜닝 후(After FT) Validation / Test 셋 평가 재실행
# -----------------------------------------------------------------------------
reranker_after_validation = reranker_validation_evaluator(reranker)
reranker_after_test = reranker_test_evaluator(reranker)
reranker_after_attribute_test = reranker_attribute_test_evaluator(reranker)

# 실수 타입 지표 파싱 헬퍼 함수
def numeric_metrics(result):
    return {key: float(value) for key, value in result.items() if isinstance(value, numbers.Real)}

# -----------------------------------------------------------------------------
# 2. Reranker 사전학습(Before) vs ETF 파인튜닝(After) 지표 비교 표 구성
# -----------------------------------------------------------------------------
comparison_rows = []
for family, before_metrics, after_metrics in [
    ("entity", reranker_before_test, reranker_after_test),
    ("attribute_challenge", reranker_before_attribute_test, reranker_after_attribute_test),
]:
    metric_names = sorted(set(before_metrics) | set(after_metrics))
    for name in metric_names:
        if isinstance(before_metrics.get(name), numbers.Real) and isinstance(after_metrics.get(name), numbers.Real):
            before_value = float(before_metrics[name])
            after_value = float(after_metrics[name])
            comparison_rows.append({"family": family, "metric": name, "before": before_value, "after": after_value, "delta": after_value - before_value})
reranker_comparison = pd.DataFrame(comparison_rows)
print(reranker_comparison.round(4).to_string(index=False))

# -----------------------------------------------------------------------------
# 3. 평가 결과 저장, Reranker 모델 로컬 저장 및 Smoke Test(스모크 테스트)
# -----------------------------------------------------------------------------
reranker_metrics = {
    "validation": {
        "before": numeric_metrics(reranker_before_validation),
        "after": numeric_metrics(reranker_after_validation),
    },
    "test": {
        "entity": {"before": numeric_metrics(reranker_before_test), "after": numeric_metrics(reranker_after_test)},
        "attribute_challenge": {"before": numeric_metrics(reranker_before_attribute_test), "after": numeric_metrics(reranker_after_attribute_test)},
    },
}
reranker_metrics_path = WORK_DIR / "reranker_eval_metrics.json"
with reranker_metrics_path.open("w", encoding="utf-8") as file:
    json.dump(reranker_metrics, file, ensure_ascii=False, indent=2)

# Reranker 파인튜닝 모델 복원 테스트 (Positive에는 0.9999, Negative에는 0.0000 연관성 출력 확인)
reranker_save_path = WORK_DIR / "bge_reranker_etf_finetuned"
reranker.save_pretrained(str(reranker_save_path))
reloaded_reranker = CrossEncoder(str(reranker_save_path), max_length=CFG["reranker_max_length"])
smoke_query_id = next(iter(ir_queries["test"]))
smoke_positive = next(iter(ir_qrels["test"][smoke_query_id]))
smoke_negative = next(doc_id for doc_id in fixed_candidates["test"][smoke_query_id] if doc_id != smoke_positive)
smoke_scores = reloaded_reranker.predict([
    [ir_queries["test"][smoke_query_id], corpus[smoke_positive]],
    [ir_queries["test"][smoke_query_id], corpus[smoke_negative]],
])
assert np.isfinite(smoke_scores).all()

if PUSH_TO_HUB:
    reranker.push_to_hub(
        f"{HF_NAMESPACE}/bge-reranker-v2-m3-etf",
        token=HF_TOKEN,
        private=HF_PRIVATE,
    )
print(f"✅ Reranker={reranker_save_path} | metrics={reranker_metrics_path} | smoke={smoke_scores}")


**실행 결과 설명 — 대부분의 이득은 어디서 왔나?**

Entity는 Stage 1부터 모든 지표가 1.0이라 추가 가치를 측정할 수 없습니다. Attribute challenge는 다음 세 단계를 구분해야 합니다.

| 단계 | nDCG@10 | 의미 |
|---|---:|---|
| Stage 1 순서 | 0.2350 | FT embedding 후보의 원래 순서 |
| Off-the-shelf Reranker | 0.7006 | pretrained reranker 적용 |
| FT Reranker | 0.7389 | ETF 추가 학습 후 |

FT의 순수 추가분은 `+0.0383`입니다. MRR은 `0.6958→0.7500`, MAP은 `0.6939→0.7426`으로 개선됐지만 attribute test가 24건뿐이므로 통계적 안정성과 누수 없는 FULL 결과가 필요합니다.

**결론**: Reranker 사용 효과는 크지만 추가 FT adapter의 승격은 보류합니다.


### **4-5. 최종 2-stage 순위 변화 읽기**

Stage 1은 빠른 embedding 점수로 20개 후보를 만들고, Stage 2는 같은 후보를 Cross-Encoder 점수로 다시 정렬합니다. `stage1=13위`가 Stage 2의 2위로 올라왔다면 Reranker가 세부 문맥을 다르게 평가했다는 뜻입니다.

이 데모에서는 gold가 Stage 1부터 1위이므로 Reranker의 추가 가치를 보여주기 어렵습니다. 실제 효과를 확인하려면 이름 없는 attribute query와 Stage 1 오답 사례를 우선 분석해야 합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 최종 2-Stage (Retriever Bi-Encoder + Reranker Cross-Encoder) 검색 데모
# -----------------------------------------------------------------------------
# 1. Part 3 파인튜닝 임베딩 모델 로드 및 300개 전체 ETF Corpus 벡터 매트릭스 계산
demo_embedding = SentenceTransformer(str(embedding_save_path))
demo_doc_ids = list(corpus)
demo_doc_matrix = demo_embedding.encode(
    [corpus[doc_id] for doc_id in demo_doc_ids],
    convert_to_numpy=True,
    normalize_embeddings=True,
)

# 2. Test 셋의 첫 번째 예시 질의 선택 및 정답(Gold) 종목 지정
query_id = next(iter(ir_queries["test"]))
query_text = ir_queries["test"][query_id]
gold_ids = ir_qrels["test"][query_id]

# 3. [Stage 1 - Bi-Encoder] 고속 코사인 유사도 연산으로 상위 Top-20 (candidate_k) 후보 추출
query_vector = demo_embedding.encode(query_text, convert_to_numpy=True, normalize_embeddings=True)
stage1_indices = np.argsort(-(demo_doc_matrix @ query_vector))[:CFG["candidate_k"]]
stage1_ids = [demo_doc_ids[index] for index in stage1_indices]

# 4. [Stage 2 - Cross-Encoder] 1차 후보 20개에 대해 [Query; Document] 세밀 연관성 점수 재정렬
pairs = [[query_text, corpus[doc_id]] for doc_id in stage1_ids]
cross_scores = reranker.predict(pairs)
stage2_ids = [stage1_ids[index] for index in np.argsort(-np.asarray(cross_scores))]

# 5. Stage 1 vs Stage 2 1~5위 검색 순위 변화 및 정답 문서 마커(✅) 대조 출력
print("Query:", query_text, "| Gold:", gold_ids)
print("\nStage 1 — Bi-Encoder")
for rank, doc_id in enumerate(stage1_ids[:5], 1):
    print(f"{rank}위 {'✅' if doc_id in gold_ids else '  '} {doc_id} | {corpus[doc_id][:100]}")
print("\nStage 2 — Cross-Encoder")
for rank, doc_id in enumerate(stage2_ids[:5], 1):
    old_rank = stage1_ids.index(doc_id) + 1
    print(f"{rank}위 {'✅' if doc_id in gold_ids else '  '} {doc_id} | stage1={old_rank}위")


**실행 결과 설명**

gold 문서는 Stage 1과 Stage 2 모두 1위입니다. 다른 후보는 Stage 1의 13·12·10·11위에서 Stage 2의 2~5위로 이동했습니다. Cross-Encoder가 후보의 상대 관련성을 크게 바꿨다는 뜻입니다.

하지만 gold가 처음부터 1위인 쉬운 사례이므로 품질 향상 사례라기보다 2-stage 파이프라인 연결 확인으로 해석합니다.


In [ ]:
# -----------------------------------------------------------------------------
# 2-Stage 검색의 Reranking 역전 효과가 선명한 고난도 Attribute Challenge 데모
# -----------------------------------------------------------------------------
# 1. Attribute Challenge Test 세트에서 1차 Bi-Encoder 랭킹 1위에 정답이 없는 어려운 질의 선택
diff_qid = None
for qid, qtext in attribute_queries["test"].items():
    g_ids = attribute_qrels["test"][qid]
    q_vec = demo_embedding.encode(qtext, convert_to_numpy=True, normalize_embeddings=True)
    s1_top = demo_doc_ids[np.argmax(demo_doc_matrix @ q_vec)]
    if s1_top not in g_ids:  # Stage 1 탑 1위가 오답인 질의 탐색
        diff_qid = qid
        break

if diff_qid:
    diff_qtext = attribute_queries["test"][diff_qid]
    diff_golds = attribute_qrels["test"][diff_qid]
    q_vec = demo_embedding.encode(diff_qtext, convert_to_numpy=True, normalize_embeddings=True)
    s1_indices = np.argsort(-(demo_doc_matrix @ q_vec))[:CFG["candidate_k"]]
    s1_ids = [demo_doc_ids[idx] for idx in s1_indices]
    
    # 2. Stage 2 Reranker 2차 재정렬
    pairs = [[diff_qtext, corpus[doc_id]] for doc_id in s1_ids]
    c_scores = reranker.predict(pairs)
    s2_ids = [s1_ids[idx] for idx in np.argsort(-np.asarray(c_scores))]
    
    print(f"=== [어려운 Attribute 질의: {diff_qtext}] ===")
    print(f"- 정답 ETF 개수: {len(diff_golds)}개")
    print("\n[Stage 1 — Bi-Encoder Top-5]")
    for rank, doc_id in enumerate(s1_ids[:5], 1):
        print(f"{rank}위 {'✅' if doc_id in diff_golds else '❌'} {doc_id} | {corpus[doc_id][:85]}...")
        
    print("\n[Stage 2 — Cross-Encoder Reranker Top-5]")
    for rank, doc_id in enumerate(s2_ids[:5], 1):
        old_rank = s1_ids.index(doc_id) + 1
        print(f"{rank}위 {'✅' if doc_id in diff_golds else '❌'} {doc_id} | Stage1 랭킹: {old_rank}위")
else:
    print("모든 test challenge 질의의 Stage 1 Top 1위가 정답입니다.")


### Part 4 실행 결과 해석 — off-the-shelf 효과 큼, FT 승격 보류

- Entity test는 Stage 1부터 1.0이라 reranker의 추가 가치를 보여주지 못했습니다.
- Attribute nDCG@10은 고정 Stage 1 순서 0.2350에서 off-the-shelf reranker 0.7006으로 크게 상승했고, FT reranker는 0.7389로 추가 +0.0383을 만들었습니다.
- MRR은 0.6958→0.7500, MAP은 0.6939→0.7426으로 추가 개선됐지만 표본은 24건뿐입니다.
- 따라서 대부분의 이득은 이미 pretrained reranker에서 왔으며, FT 추가 이득은 paired bootstrap과 누수 없는 FULL test 전에는 승격 근거가 아닙니다.
- smoke score 0.99998/0.00001은 해당 positive/negative 쌍의 분리를 보여줄 뿐 보정된 확률은 아닙니다. 기존 exact-name demo도 reranking의 실패 사례를 충분히 보여주지 못합니다.


## **전체 결과 판정표**

| Part | 실행 | 품질 판정 | 근거 | 다음 행동 |
|---|---|---|---|---|
| 1. SFT | 성공 | `FAIL` | ROUGE +0.0077보다 반복·환각 악화가 큼 | 데이터·LR·epoch·종료 조건 재검토 |
| 2. Tool Calling | 성공 | 개선, 일반화 주의 | Challenge execution 0.35→0.75, 오류 25% 잔존 | adversarial/실사용 사례 확대 |
| 3. Embedding | 성공 | `FAIL` | attribute nDCG 0.4311→0.2352 | pretrained BGE-M3 fallback |
| 4. Reranker | 성공 | FT 승격 보류 | pretrained 0.7006, FT 0.7389, n=24 | validation gate·bootstrap·FULL 평가 |

### **파이프라인 관점의 핵심**

`질문 → Tool route 또는 검색 → Embedding 후보 생성 → Reranker 정렬 → 근거 기반 답변`

뒤 단계가 좋아도 앞 단계가 정답을 잃으면 복구할 수 없습니다. 따라서 Tool Calling은 routing/argument/execution을, Embedding은 candidate Recall을, Reranker는 고정 후보 내부 nDCG를 각각 분리해서 봅니다.


---

## 실습 문제

### 실습 1: SFT 데이터 품질 규칙
KoAlpaca 필터 기준 하나를 변경하고 reject 통계, test ROUGE-L, factual diagnostic, 반복률 변화를 비교하세요.

### 실습 2: ETF Tool Calling 확장
`search_etfs`에 `replication_method` 필터를 추가하고 Template/Challenge 양쪽의 schema validation과 execution accuracy를 갱신하세요.

### 실습 3: Text2SQL 정책 테스트
허용 집계 3개와 거부 SQL 5개(쓰기, JOIN, 상세 행, 잘못된 열, 다중 statement)를 만들고 gateway 결과를 설명하세요.

### 실습 4: Embedding hard negative
random negative와 baseline top hard negative로 각각 학습해 entity 및 attribute challenge nDCG@10을 비교하세요.

### 실습 5: Reranker candidate K
`candidate_k=10/20/50`에서 Stage 1 Recall@K와 Stage 2 nDCG@10, latency를 함께 측정하세요.


### **실습을 진행하는 방법**

아래 빈 코드 셀은 원본 실습 공간입니다. 한 번에 모든 실습을 수행하기보다 다음 순서로 진행하세요.

1. 변경 전 지표 JSON과 설정을 별도 이름으로 저장합니다.
2. 변수 하나만 바꾸고 같은 split·seed·생성 옵션을 유지합니다.
3. validation에서 가설을 확인합니다.
4. 마지막 한 번만 잠긴 test를 보고합니다.
5. 점수뿐 아니라 실패 사례와 VRAM·지연 시간도 기록합니다.

> ⚠️ `[Test]`에 저장된 기존 출력은 역사적 증거이므로 이 가이드 파일에서 다시 실행해 덮어쓰기보다 새 실행 결과 파일을 만들어 비교하는 편이 안전합니다.


In [ ]:
# 여기에 실습 코드를 작성하세요.


<details close>
<summary>💡 실습별 확인 포인트</summary>

1. **SFT 데이터 품질**: 변경한 필터로 reject 수가 어떻게 달라졌는지, validation loss와 반복률이 같은 방향인지 설명합니다.
2. **Tool schema 확장**: 새 인자가 schema 검사뿐 아니라 실제 SQLite 실행과 Challenge gold에도 반영됐는지 확인합니다.
3. **SQL 정책**: 허용 집계는 결과 행까지 검증하고, 쓰기·JOIN·상세 행·잘못된 열·다중 문장이 각각 어느 방어선에서 거부됐는지 기록합니다.
4. **Embedding negative**: entity 점수만 보지 말고 attribute nDCG/Recall과 대표 실패 query를 비교합니다.
5. **Candidate K**: K가 커질 때 Stage 1 Recall, Stage 2 nDCG, pair 수와 latency가 어떻게 함께 변하는지 표로 정리합니다.

**완료 체크**: 동일 seed/split, validation 기반 선택, test 1회 보고, Before/After 설정 동일, 결과 JSON과 실패 사례 보존.

</details>


---

## 정리

1. Glaive는 범용 Tool Calling의 구조와 정제 문제를 보여주는 사례입니다.
2. 도메인 학습 데이터는 실제 ETF 값에서 gold plan을 먼저 만들고 실행 검증한 뒤 합성해야 합니다.
3. Tool Calling은 routing·argument·execution을, Embedding은 candidate recall을, Reranker는 candidate 내부 순위를 평가합니다.
4. Reranker가 좋아도 Stage 1이 정답을 가져오지 못하면 복구할 수 없으므로 Recall@K와 nDCG를 함께 봅니다.
5. Before/After 비교는 같은 split, 같은 후보, greedy decoding으로 재현 가능하게 수행합니다.
6. Template 점수의 1.0 포화는 완료 신호가 아닙니다. 이름 없는 속성 질의와 compositional challenge로 일반화 여지를 계속 확인합니다.

### 참고 자료

- [Unsloth](https://github.com/unslothai/unsloth)
- [Glaive Function Calling v2](https://huggingface.co/datasets/glaiveai/glaive-function-calling-v2)
- [BGE-M3](https://huggingface.co/BAAI/bge-m3)
- [BGE Reranker v2 M3](https://huggingface.co/BAAI/bge-reranker-v2-m3)
- [Sentence Transformers — Cross-Encoder Training](https://www.sbert.net/docs/cross_encoder/training_overview.html)
- [Sentence Transformers — Cross-Encoder Evaluation](https://www.sbert.net/docs/package_reference/cross_encoder/evaluation.html)
- [sqlglot](https://github.com/tobymao/sqlglot)
- Kusupati et al., *Matryoshka Representation Learning*, NeurIPS 2022


## **초보자 핵심 용어집**

| 용어 | 한 문장 설명 |
|---|---|
| SFT | 질문과 모범 답변을 이용해 모델의 응답 행동을 지도 학습하는 방법 |
| QLoRA | 양자화한 원본 모델은 고정하고 작은 LoRA 행렬만 학습하는 방법 |
| Response-only loss | assistant가 생성할 응답 토큰에만 학습 오차를 적용하는 방식 |
| Tool Calling | 모델이 함수를 직접 실행하지 않고 도구 이름과 JSON 인자를 제안하는 방식 |
| Schema | 허용되는 입력 필드·자료형·필수값을 정의한 계약 |
| Embedding | 텍스트 의미를 고정 길이 벡터로 표현한 값 |
| Hard negative | 정답과 비슷해 모델이 헷갈리지만 실제로는 오답인 학습 문서 |
| Bi-Encoder | query와 document를 독립적으로 인코딩하는 빠른 검색 모델 |
| Cross-Encoder | query와 document를 함께 읽어 정밀 점수를 내는 모델 |
| Promotion gate | 새 모델을 실제 사용 후보로 인정하기 위해 통과해야 하는 검증 기준 |

### **마지막으로 기억할 세 문장**

1. 코드가 끝까지 실행됐다는 사실은 모델 품질이 좋아졌다는 뜻이 아닙니다.
2. 쉬운 표본의 1.0보다 어려운 독립 Challenge의 실패를 먼저 분석합니다.
3. validation으로 선택하고 잠긴 test는 마지막 보고에만 사용해야 합니다.
